# 03 - KPIs Financeiros

Este notebook apresenta o cálculo dos principais indicadores financeiros utilizados na análise dos **1.200 cenários ao longo de 12 anos**.

São analisados indicadores de **rentabilidade, liquidez, estrutura de capital e geração de valor**, que posteriormente são utilizados na comparação e construção do ranking financeiro.

##  Carregamento da base tratada

A base utilizada nesta etapa é o arquivo gerado no tratamento dos dados.

In [1]:
import pandas as pd

caminho = "../staging/dados_tratados.csv"

df_kpis = pd.read_csv(
    caminho,
    sep=";",
    encoding="utf-8-sig"
)

print("Dimensões da base:", df_kpis.shape)
print("Quantidade de cenários:", df_kpis["Cenario"].nunique())
print("Quantidade de anos:", df_kpis["Ano"].nunique())
print("Quantidade de contas:", df_kpis["Conta"].nunique())

display(df_kpis.head())

Dimensões da base: (958800, 4)
Quantidade de cenários: 1200
Quantidade de anos: 12
Quantidade de contas: 69


,Cenario,Ano,Conta,Valor_numerico
0,Total Cen_00001,Ano 1,BAL - Amortização - Intangível,-5.047124e+09
1,Total Cen_00001,Ano 1,BAL - Amortização Acumulada,-3.011128e+07
2,Total Cen_00001,Ano 1,BAL - At Fiscal Diferido,-1.039693e+08
3,Total Cen_00001,Ano 1,BAL - Ativo Circulante,1.795721e+09
4,Total Cen_00001,Ano 1,BAL - Capital Social,-3.096189e+08


##  Consulta de um cenário

Nesta etapa, vamos selecionar um cenário e organizar suas contas por ano.

In [6]:
cenario_selecionado = "Total Cen_00001"

consulta = (
    df_kpis[df_kpis["Cenario"] == cenario_selecionado]
    .pivot_table(
        index="Conta",
        columns="Ano",
        values="Valor_numerico",
        aggfunc="first"
    )
)

ordem_anos = [f"Ano {i}" for i in range(1, 13)]

consulta = consulta.reindex(columns=ordem_anos)

print("Cenário selecionado:", cenario_selecionado)

display(consulta)

Cenário selecionado: Total Cen_00001


Ano,Ano 1,Ano 2,Ano 3,Ano 4,Ano 5,Ano 6,Ano 7,Ano 8,Ano 9,Ano 10,Ano 11,Ano 12
Conta,,,,,,,,,,,,
BAL - Amortização - Intangível,-5.047124e+09,-5.710884e+09,-6.417036e+09,-7.165162e+09,-7.936959e+09,-8.723575e+09,-9.525109e+09,-1.034131e+10,-1.116491e+10,-1.200028e+10,-1.284384e+10,-2.990000e-07
BAL - Amortização Acumulada,-3.011128e+07,-4.054538e+07,-5.334893e+07,-6.756197e+07,-8.255313e+07,-1.006275e+08,-1.215114e+08,-1.448155e+08,-1.806699e+08,-2.234075e+08,-2.746443e+08,NaN
BAL - At Fiscal Diferido,-1.039693e+08,1.636680e+08,4.649431e+08,6.293188e+08,6.229204e+08,8.631238e+08,1.201250e+09,1.573346e+09,1.749407e+09,1.688779e+09,1.859161e+09,7.500000e-08
BAL - Ativo Circulante,1.795721e+09,1.567979e+09,2.292584e+09,1.655697e+09,8.393786e+08,7.907685e+08,9.407401e+08,1.779399e+09,2.208646e+09,2.461672e+09,3.209590e+09,3.877167e+08
BAL - Capital Social,-3.096189e+08,-3.096189e+08,-3.096189e+08,-3.096189e+08,-3.096189e+08,-3.096189e+08,-3.096189e+08,-3.096189e+08,-3.096189e+08,-3.096189e+08,-3.096189e+08,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
FLU - Receita,5.396831e+09,5.872798e+09,6.123728e+09,6.443489e+09,6.812088e+09,7.237617e+09,7.710044e+09,8.242633e+09,8.742053e+09,9.313009e+09,9.918856e+09,1.345478e+10
FLU - Resultado Financeiro,-7.304044e+08,-5.457156e+08,-4.887332e+08,-1.580896e+09,-1.525871e+09,-1.112855e+09,-1.034734e+09,-3.344773e+08,-2.160440e+08,-1.373751e+08,-8.781890e+07,2.205148e+07
FLU - Saldo Final,1.205265e+09,1.022427e+09,1.748421e+09,1.117390e+09,3.073864e+08,2.661750e+08,4.276930e+08,1.369964e+09,1.923333e+09,2.325378e+09,3.245810e+09,4.181949e+08


## Separação das demonstrações

A consulta do cenário contém contas do Balanço Patrimonial (BAL), da Demonstração do Resultado (DRE) e do Fluxo de Caixa (FLU).

Vamos separar essas três demonstrações para facilitar a organização e as análises posteriores.

In [ ]:
bal = consulta[consulta.index.str.startswith("BAL -")].copy()
dre = consulta[consulta.index.str.startswith("DRE -")].copy()
flu = consulta[consulta.index.str.startswith("FLU -")].copy()

print("Quantidade de contas no BAL:", len(bal))
print("Quantidade de contas na DRE:", len(dre))
print("Quantidade de contas no FLU:", len(flu))

## Conferência das contas

Antes de organizar as demonstrações, vamos conferir quais contas existem em cada uma delas.

In [ ]:
print("Contas do Balanço Patrimonial:")
display(bal.index.to_frame(name="Conta").reset_index(drop=True))

print("Contas da DRE:")
display(dre.index.to_frame(name="Conta").reset_index(drop=True))

print("Contas do Fluxo de Caixa:")
display(flu.index.to_frame(name="Conta").reset_index(drop=True))

## Estrutura das contas por ano

As contas do Balanço Patrimonial não aparecem da mesma forma em todos os anos.

Vamos verificar a quantidade de contas disponíveis em cada demonstração para cada ano do cenário selecionado.

In [ ]:
estrutura = pd.DataFrame({
    "BAL": bal.notna().sum(),
    "DRE": dre.notna().sum(),
    "FLU": flu.notna().sum()
})

display(estrutura)

## Variação das contas do Balanço Patrimonial

O Balanço Patrimonial possui contas que aparecem ou deixam de aparecer em determinados anos.
Aqui estamos contando, para cada conta do BAL, em quantos dos 12 anos ela possui um valor.

In [ ]:
contas_bal_por_ano = bal.notna().sum(axis=1)

contas_variaveis = contas_bal_por_ano[
    contas_bal_por_ano != 12
].sort_values()

display(contas_variaveis.to_frame("Anos com valor"))

## Identificando os anos sem valor

Verificando em quais anos as contas variáveis do Balanço Patrimonial não possuem informação.

In [ ]:
contas_variaveis = contas_variaveis.index

anos_sem_valor = bal.loc[contas_variaveis].isna()

display(anos_sem_valor)

## Contas disponíveis nas demonstrações

Vamos listar as contas de cada demonstração para definir a organização que será utilizada na análise.

In [ ]:
contas_bal = bal.index.tolist()
contas_dre = dre.index.tolist()
contas_flu = flu.index.tolist()

print("BAL:", contas_bal)
print("DRE:", contas_dre)
print("FLU:", contas_flu)

## Organização da DRE

Vamos organizar as contas da DRE em uma ordem mais próxima da apresentação contábil, mantendo apenas as contas existentes na base.

In [ ]:
ordem_dre = [
    "DRE - Receita",
    "DRE - Tributos",
    "DRE - Custos",
    "DRE - Depreciação e Amortização",
    "DRE - Resultado Operacional",
    "DRE - Outros Resultados Operacionais",
    "DRE - Resultado Financeiro",
    "DRE - Resultado Antes do Imposto de Renda",
    "DRE - Imposto de Renda e Contribuição Social",
    "DRE - Resultado Líquido",
    "DRE - Resultado Líquido após Equivalência",
    "DRE - Receitas Financeiras",
    "DRE - Despesas Financeiras",
    "DRE - EBITDA"
]

dre_organizada = dre.reindex(ordem_dre)

display(dre_organizada)

## Organização do Balanço Patrimonial

Vamos separar as contas do Balanço Patrimonial em Ativo, Passivo e Patrimônio Líquido, mantendo os nomes e valores originais da base.

In [ ]:
contas_ativo = [
    conta for conta in contas_bal
    if any(palavra in conta for palavra in [
        "Ativo",
        "Disponível",
        "Contas a Receber",
        "Créditos",
        "Estoques",
        "Investimentos",
        "Outorga",
        "Diferido",
        "Amortização",
        "Depreciação",
        "Realizável",
        "Permanente"
    ])
]

contas_passivo = [
    conta for conta in contas_bal
    if any(palavra in conta for palavra in [
        "Passivo",
        "Fornecedores",
        "Emprést",
        "Tributos",
        "Impostos",
        "Encargos",
        "Obrigações",
        "Provisão",
        "Outros Débitos",
        "Outros deb",
        "Contas a Pagar",
        "Exigível"
    ])
]

contas_pl = [
    conta for conta in contas_bal
    if any(palavra in conta for palavra in [
        "Patrimônio Líquido",
        "Capital Social",
        "Reservas",
        "Resultado",
        "Dividendos"
    ])
]

print("Ativo:")
display(pd.DataFrame({"Conta": contas_ativo}))

print("Passivo:")
display(pd.DataFrame({"Conta": contas_passivo}))

print("Patrimônio Líquido:")
display(pd.DataFrame({"Conta": contas_pl}))

## Organização do Balanço Patrimonial

Agora vamos definir a ordem das contas do BAL para apresentar o Balanço Patrimonial de forma organizada, mantendo os valores originais da base.

In [ ]:
ordem_bal = [
    # Ativo
    "BAL - Ativo Circulante",
    "BAL - Disponível",
    "BAL - Contas a Receber - Clientes",
    "BAL - Contas a Receber - Partes Relacionadas",
    "BAL - Contas a Receber - SWAP",
    "BAL - Estoques Diversos",
    "BAL - Créditos Tributários",
    "BAL - Outros Créditos",
    "BAL - Realizável a Longo Prazo",
    "BAL - Outros Créditos LP",
    "BAL - At Fiscal Diferido",
    "BAL - Investimentos - Imobilizado",
    "BAL - Investimentos - Intangível",
    "BAL - Outorga da Concessão",
    "BAL - Diferido",
    "BAL - Amortização - Intangível",
    "BAL - Amortização Acumulada",
    "BAL - Depreciação Acumulada",
    "BAL - Permanente",
    "BAL - Total do Ativo",

    # Passivo
    "BAL - Passivo Circulante",
    "BAL - Fornecedores",
    "BAL - Contas a Pagar - Parte Relacionada",
    "BAL - Emprést",
    "BAL - Empréstimos",
    "BAL - Tributos a pagar",
    "BAL - Impostos",
    "BAL - Encargos Sociais e Trabalhistas",
    "BAL - Obrigações com o Poder Concedente",
    "BAL - Provisão Manutenção",
    "BAL - Prov para Contingências",
    "BAL - Outros Débitos",
    "BAL - Outros deb",
    "BAL - Exigível a Longo Prazo",
    "BAL - Total do Passivo",

    # Patrimônio Líquido
    "BAL - Patrimônio Líquido",
    "BAL - Capital Social",
    "BAL - Reservas Legais",
    "BAL - Reservas de Capital",
    "BAL - Reserva de Retenção de Lucros",
    "BAL - Resultado Acumulado",
    "BAL - Resultado do Período",
    "BAL - Dividendos Antecipados"
]

bal_organizado = bal.reindex(ordem_bal)

print("Quantidade de contas no BAL:", len(bal_organizado))

display(bal_organizado)

## Organização do Fluxo de Caixa

Agora vamos organizar as contas do Fluxo de Caixa (FLU) na ordem da demonstração, mantendo os valores originais da base.

In [ ]:
ordem_flu = [
    "FLU - Entradas",
    "FLU - Receita",
    "FLU - Tributos",
    "FLU - Custos",
    "FLU - Resultado Financeiro",
    "FLU - Despesas Financeiras",
    "FLU - Imposto de Renda e Contribuição Social",
    "FLU - Geração de Caixa",
    "FLU - Investimentos",
    "FLU - Distribuição para Acionista",
    "FLU - Saldo Inicial",
    "FLU - Saldo Final"
]

flu_organizado = flu.reindex(ordem_flu)

display(flu_organizado)

## Validação final das demonstrações

Antes de calcular os indicadores financeiros, vamos conferir se as demonstrações estão estruturadas corretamente.

Nesta etapa, vamos verificar se BAL, DRE e FLU possuem os anos esperados e se não existem contas duplicadas dentro do cenário selecionado.

In [ ]:
print("Validação das demonstrações")

print("\nBAL")
print("Contas:", len(bal_organizado))
print("Anos:", bal_organizado.shape[1])
print("Contas duplicadas:", bal_organizado.index.duplicated().sum())

print("\nDRE")
print("Contas:", len(dre_organizada))
print("Anos:", dre_organizada.shape[1])
print("Contas duplicadas:", dre_organizada.index.duplicated().sum())

print("\nFLU")
print("Contas:", len(flu_organizado))
print("Anos:", flu_organizado.shape[1])
print("Contas duplicadas:", flu_organizado.index.duplicated().sum())

print("\nAnos disponíveis:")
print(list(bal_organizado.columns))

## Conferência das contas do Balanço Patrimonial

Vamos comparar as contas existentes no BAL original com a lista utilizada para organizar a demonstração.

O objetivo é identificar se alguma conta ficou de fora durante a organização.

In [ ]:
contas_bal_originais = set(bal.index)
contas_bal_organizadas = set(bal_organizado.index)

faltantes = contas_bal_originais - contas_bal_organizadas

print("Contas no BAL original:", len(contas_bal_originais))
print("Contas no BAL organizado:", len(contas_bal_organizadas))

print("\nConta(s) que ficaram de fora:")
for conta in sorted(faltantes):
    print("-", conta)

## Validação do Balanço Patrimonial

Agora vamos verificar algumas relações contábeis do Balanço Patrimonial.

O objetivo é identificar se os valores fornecidos pela base são coerentes entre si antes de utilizá-los no cálculo dos indicadores financeiros.

In [ ]:
ativo = bal_organizado.loc["BAL - Total do Ativo"]
passivo = bal_organizado.loc["BAL - Total do Passivo"]

diferenca_bal = ativo + passivo

print("Validação do Balanço Patrimonial")
print("\nDiferença entre Total do Ativo e Total do Passivo:")

display(diferenca_bal.to_frame("Diferença"))

## Validação da composição do Ativo

Vamos verificar se o Total do Ativo é compatível com a soma do Ativo Circulante, do Realizável a Longo Prazo e do Permanente.

Pequenas diferenças podem ocorrer devido ao arredondamento dos valores fornecidos pela base.

In [ ]:
ativo_circulante = bal_organizado.loc["BAL - Ativo Circulante"]
realizavel_lp = bal_organizado.loc["BAL - Realizável a Longo Prazo"]
permanente = bal_organizado.loc["BAL - Permanente"]
total_ativo = bal_organizado.loc["BAL - Total do Ativo"]

diferenca_ativo = (
    ativo_circulante
    + realizavel_lp
    + permanente
    - total_ativo
)

print("Validação da composição do Ativo")
print("\nDiferença entre a composição calculada e o Total do Ativo:")

display(diferenca_ativo.to_frame("Diferença"))

## Validação do Patrimônio Líquido

Vamos verificar se o Patrimônio Líquido apresentado na base é compatível com a soma das suas principais contas componentes.

In [ ]:
capital_social = bal_organizado.loc["BAL - Capital Social"]
reservas_legais = bal_organizado.loc["BAL - Reservas Legais"]
reservas_capital = bal_organizado.loc["BAL - Reservas de Capital"]
reserva_retencao = bal_organizado.loc["BAL - Reserva de Retenção de Lucros"]
resultado_acumulado = bal_organizado.loc["BAL - Resultado Acumulado"]
resultado_periodo = bal_organizado.loc["BAL - Resultado do Período"]
dividendos_antecipados = bal_organizado.loc["BAL - Dividendos Antecipados"]

pl_calculado = (
    capital_social
    + reservas_legais
    + reservas_capital
    + reserva_retencao
    + resultado_acumulado
    + resultado_periodo
    + dividendos_antecipados
)

pl_informado = bal_organizado.loc["BAL - Patrimônio Líquido"]

diferenca_pl = pl_calculado - pl_informado

print("Validação do Patrimônio Líquido")
display(diferenca_pl.to_frame("Diferença"))

## Verificação das contas do Patrimônio Líquido

Antes de continuar as validações, vamos verificar quais contas do Patrimônio Líquido estão disponíveis em cada ano.

Isso ajuda a diferenciar contas realmente zeradas de contas que não fazem parte da estrutura do ano.

In [ ]:
contas_pl = [
    "BAL - Capital Social",
    "BAL - Reservas Legais",
    "BAL - Reservas de Capital",
    "BAL - Reserva de Retenção de Lucros",
    "BAL - Resultado Acumulado",
    "BAL - Resultado do Período",
    "BAL - Dividendos Antecipados"
]

estrutura_pl = bal_organizado.loc[contas_pl].notna()

print("Contas do Patrimônio Líquido disponíveis por ano:")
display(estrutura_pl)

## Validação do Resultado Operacional

Vamos verificar se o Resultado Operacional informado na DRE é compatível com a soma das contas que o compõem.

In [ ]:
receita = dre_organizada.loc["DRE - Receita"]
tributos = dre_organizada.loc["DRE - Tributos"]
custos = dre_organizada.loc["DRE - Custos"]
da = dre_organizada.loc["DRE - Depreciação e Amortização"]

resultado_operacional = dre_organizada.loc["DRE - Resultado Operacional"]

resultado_operacional_calculado = (
    receita
    + tributos
    + custos
    + da
)

diferenca_dre = (
    resultado_operacional_calculado
    - resultado_operacional
)

print("Validação do Resultado Operacional")
display(diferenca_dre.to_frame("Diferença"))

## Validação do Resultado Antes do Imposto de Renda

Vamos verificar se o Resultado Antes do Imposto de Renda é compatível com o Resultado Operacional, os Outros Resultados Operacionais e o Resultado Financeiro.

In [ ]:
resultado_operacional = dre_organizada.loc["DRE - Resultado Operacional"]
outros_resultados = dre_organizada.loc["DRE - Outros Resultados Operacionais"]
resultado_financeiro = dre_organizada.loc["DRE - Resultado Financeiro"]

resultado_antes_ir = dre_organizada.loc[
    "DRE - Resultado Antes do Imposto de Renda"
]

resultado_antes_ir_calculado = (
    resultado_operacional
    + outros_resultados
    + resultado_financeiro
)

diferenca_antes_ir = (
    resultado_antes_ir_calculado
    - resultado_antes_ir
)

print("Validação do Resultado Antes do Imposto de Renda")
display(diferenca_antes_ir.to_frame("Diferença"))

## Validação do Resultado Líquido

Vamos verificar se o Resultado Líquido informado na DRE é compatível com o Resultado Antes do Imposto de Renda e o Imposto de Renda e Contribuição Social.

In [ ]:
resultado_antes_ir = dre_organizada.loc[
    "DRE - Resultado Antes do Imposto de Renda"
]

ir_cs = dre_organizada.loc[
    "DRE - Imposto de Renda e Contribuição Social"
]

resultado_liquido = dre_organizada.loc[
    "DRE - Resultado Líquido"
]

resultado_liquido_calculado = (
    resultado_antes_ir
    + ir_cs
)

diferenca_resultado_liquido = (
    resultado_liquido_calculado
    - resultado_liquido
)

print("Validação do Resultado Líquido")
display(
    diferenca_resultado_liquido.to_frame("Diferença")
)

## Comparação entre EBITDA e Resultado Operacional

O EBITDA já é informado diretamente pela base.

Nesta etapa, vamos apenas calcular a diferença entre o EBITDA e o Resultado Operacional para entender a relação entre os dois indicadores, sem alterar os valores originais.

In [ ]:
ebitda = dre_organizada.loc["DRE - EBITDA"]
resultado_operacional = dre_organizada.loc[
    "DRE - Resultado Operacional"
]

diferenca_ebitda = ebitda - resultado_operacional

print("Diferença entre EBITDA e Resultado Operacional")
display(
    diferenca_ebitda.to_frame("Diferença")
)

## Validação do Resultado Financeiro

Nesta etapa, vamos verificar se o Resultado Financeiro informado na base é compatível com as Receitas Financeiras e as Despesas Financeiras.

In [ ]:
receitas_financeiras = dre_organizada.loc[
    "DRE - Receitas Financeiras"
]

despesas_financeiras = dre_organizada.loc[
    "DRE - Despesas Financeiras"
]

resultado_financeiro = dre_organizada.loc[
    "DRE - Resultado Financeiro"
]

resultado_financeiro_calculado = (
    receitas_financeiras
    + despesas_financeiras
)

diferenca_resultado_financeiro = (
    resultado_financeiro_calculado
    - resultado_financeiro
)

print("Validação do Resultado Financeiro")
display(
    diferenca_resultado_financeiro.to_frame("Diferença")
)

## Validação do Resultado Líquido após Equivalência

Vamos comparar o Resultado Líquido com o Resultado Líquido após Equivalência para verificar como esses valores se relacionam na base.

In [ ]:
resultado_liquido = dre_organizada.loc[
    "DRE - Resultado Líquido"
]

resultado_liquido_equivalencia = dre_organizada.loc[
    "DRE - Resultado Líquido após Equivalência"
]

diferenca_equivalencia = (
    resultado_liquido_equivalencia
    - resultado_liquido
)

print("Diferença entre Resultado Líquido após Equivalência e Resultado Líquido")

display(
    diferenca_equivalencia.to_frame("Diferença")
)

## Validação do Saldo Final do FLU

Nesta etapa, vamos verificar se o Saldo Final do FLU é compatível com o Saldo Inicial e as movimentações de caixa apresentadas na demonstração.

In [ ]:
saldo_inicial = flu_organizado.loc[
    "FLU - Saldo Inicial"
]

geracao_caixa = flu_organizado.loc[
    "FLU - Geração de Caixa"
]

investimentos = flu_organizado.loc[
    "FLU - Investimentos"
]

distribuicao = flu_organizado.loc[
    "FLU - Distribuição para Acionista"
]

saldo_final = flu_organizado.loc[
    "FLU - Saldo Final"
]

saldo_final_calculado = (
    saldo_inicial
    + geracao_caixa
    + investimentos
    + distribuicao
)

diferenca_saldo_final = (
    saldo_final_calculado
    - saldo_final
)

print("Validação do Saldo Final do FLU")

display(
    diferenca_saldo_final.to_frame("Diferença")
)

## Estrutura do FLU

Antes de validar os cálculos do fluxo de caixa, vamos visualizar todas as contas do FLU do Cenário 1 ao longo dos 12 anos.

Isso permite identificar como a demonstração está estruturada antes de definir as relações que serão validadas.

In [ ]:
id="4xq8sm"
print("FLU do cenário:", cenario_selecionado)

display(flu_organizado)

## Relação entre Saldo Inicial e Saldo Final

Vamos verificar como o saldo de caixa evolui de um ano para o outro.

Primeiro, vamos comparar o Saldo Inicial de cada ano com o Saldo Final do ano anterior. Essa relação deve ajudar a identificar a lógica utilizada pelo fluxo de caixa.

In [ ]:
saldo_inicial = flu_organizado.loc[
    "FLU - Saldo Inicial"
]

saldo_final = flu_organizado.loc[
    "FLU - Saldo Final"
]

comparacao_saldos = pd.DataFrame({
    "Saldo Inicial": saldo_inicial,
    "Saldo Final": saldo_final,
    "Saldo Final do Ano Anterior": saldo_final.shift(1)
})

print("Relação entre os saldos do FLU:")

display(comparacao_saldos)

## Variação do saldo de caixa

Agora vamos calcular quanto o saldo de caixa aumentou ou diminuiu em cada ano.

Essa variação será comparada com as linhas de movimentação do FLU para identificar como o Saldo Final é formado.

In [ ]:
saldo_inicial = flu_organizado.loc[
    "FLU - Saldo Inicial"
]

saldo_final = flu_organizado.loc[
    "FLU - Saldo Final"
]

variacao_saldo = saldo_final - saldo_inicial

print("Variação do saldo de caixa:")

display(
    variacao_saldo.to_frame("Variação do Saldo")
)

## Validação do Saldo Final do FLU

A análise anterior mostrou que a variação do saldo de caixa é igual à Geração de Caixa.

Portanto, vamos validar se:

Saldo Final = Saldo Inicial + Geração de Caixa

In [ ]:
saldo_inicial = flu_organizado.loc[
    "FLU - Saldo Inicial"
]

geracao_caixa = flu_organizado.loc[
    "FLU - Geração de Caixa"
]

saldo_final = flu_organizado.loc[
    "FLU - Saldo Final"
]

saldo_final_calculado = (
    saldo_inicial
    + geracao_caixa
)

diferenca_saldo_final = (
    saldo_final_calculado
    - saldo_final
)

print("Validação do Saldo Final do FLU:")

display(
    diferenca_saldo_final.to_frame("Diferença")
)

## Índices de Liquidez

Os índices de liquidez mostram a capacidade financeira da empresa de cumprir suas obrigações.

Nesta etapa, vamos calcular:
- Liquidez Corrente (LC)
- Liquidez Seca (LS)
- Liquidez Imediata (LI)
- Liquidez Geral (LG)

Os cálculos serão feitos para os 12 anos do cenário selecionado.

In [ ]:
# Contas utilizadas nos índices de liquidez

ativo_circulante = bal_organizado.loc[
    "BAL - Ativo Circulante"
]

disponivel = bal_organizado.loc[
    "BAL - Disponível"
]

estoques = bal_organizado.loc[
    "BAL - Estoques Diversos"
]

realizavel_lp = bal_organizado.loc[
    "BAL - Realizável a Longo Prazo"
]

passivo_circulante = bal_organizado.loc[
    "BAL - Passivo Circulante"
]

exigivel_lp = bal_organizado.loc[
    "BAL - Exigível a Longo Prazo"
]

pc = passivo_circulante.abs()
el = exigivel_lp.abs()

# Índices de liquidez

lc = ativo_circulante / pc

ls = (ativo_circulante - estoques) / pc

li = disponivel / pc

lg = (ativo_circulante + realizavel_lp) / (pc + el)

# Tabela final dos índices

indices_liquidez = pd.DataFrame({
    "Liquidez Corrente (LC)": lc,
    "Liquidez Seca (LS)": ls,
    "Liquidez Imediata (LI)": li,
    "Liquidez Geral (LG)": lg
})

print("Índices de liquidez —", cenario_selecionado)

display(indices_liquidez)

## Índices de Endividamento

Os índices de endividamento mostram como a empresa está financiando seus ativos e qual é a participação do capital de terceiros.

Nesta etapa, vamos calcular:
- Participação de Capital de Terceiros (PCT)
- Composição do Endividamento (CE)
- Imobilização do Patrimônio Líquido (IPL)

In [ ]:
# Contas utilizadas

passivo_circulante = bal_organizado.loc[
    "BAL - Passivo Circulante"
].abs()

exigivel_lp = bal_organizado.loc[
    "BAL - Exigível a Longo Prazo"
].abs()

patrimonio_liquido = bal_organizado.loc[
    "BAL - Patrimônio Líquido"
].abs()

permanente = bal_organizado.loc[
    "BAL - Permanente"
]

# Capital de terceiros
capital_terceiros = passivo_circulante + exigivel_lp

# Índices de endividamento

pct = capital_terceiros / patrimonio_liquido

ce = passivo_circulante / capital_terceiros

ipl = permanente / patrimonio_liquido

# Tabela

indices_endividamento = pd.DataFrame({
    "PCT": pct,
    "CE": ce,
    "IPL": ipl
})

print("Índices de endividamento —", cenario_selecionado)

display(indices_endividamento)

## Validação do EBITDA por engenharia reversa

O EBITDA já é informado na base original.

Para validar esse indicador, vamos comparar o EBITDA informado com diferentes combinações das contas da DRE, buscando identificar a composição utilizada no modelo original.

A validação será feita sem alterar o EBITDA fornecido pela empresa.

In [ ]:
# Contas da DRE

receita = dre_organizada.loc["DRE - Receita"]
tributos = dre_organizada.loc["DRE - Tributos"]
custos = dre_organizada.loc["DRE - Custos"]
da = dre_organizada.loc["DRE - Depreciação e Amortização"]

resultado_operacional = dre_organizada.loc[
    "DRE - Resultado Operacional"
]

outros_resultados = dre_organizada.loc[
    "DRE - Outros Resultados Operacionais"
]

ebitda_informado = dre_organizada.loc[
    "DRE - EBITDA"
]

# Possíveis reconstruções

ebitda_1 = resultado_operacional - da

ebitda_2 = receita + tributos + custos

ebitda_3 = resultado_operacional + (-da)

ebitda_4 = (
    resultado_operacional
    + (-da)
    + outros_resultados
)

# Comparação das diferenças

validacao_ebitda = pd.DataFrame({
    "EBITDA informado": ebitda_informado,
    "RO - D&A": ebitda_1,
    "Receita + Tributos + Custos": ebitda_2,
    "RO + (-D&A)": ebitda_3,
    "RO + (-D&A) + Outros Resultados": ebitda_4
})

print("Engenharia reversa do EBITDA —", cenario_selecionado)

display(validacao_ebitda)

## Engenharia reversa do EBITDA

As primeiras reconstruções não reproduziram o EBITDA informado na base.

Agora vamos calcular a diferença entre o EBITDA informado e o resultado operacional ajustado, para identificar quais contas da DRE podem explicar essa diferença.

In [ ]:
# EBITDA informado
ebitda = dre_organizada.loc[
    "DRE - EBITDA"
]

# Resultado operacional
resultado_operacional = dre_organizada.loc[
    "DRE - Resultado Operacional"
]

# Depreciação e amortização
da = dre_organizada.loc[
    "DRE - Depreciação e Amortização"
]

# Outros resultados operacionais
outros_resultados = dre_organizada.loc[
    "DRE - Outros Resultados Operacionais"
]

# Reconstrução mais próxima encontrada até agora
ebitda_reconstruido = (
    resultado_operacional
    - da
    + outros_resultados
)

# Diferença que ainda precisamos explicar
diferenca = ebitda - ebitda_reconstruido

engenharia_ebitda = pd.DataFrame({
    "EBITDA informado": ebitda,
    "EBITDA reconstruído": ebitda_reconstruido,
    "Diferença": diferenca
})

print("Engenharia reversa do EBITDA —", cenario_selecionado)

display(engenharia_ebitda)

## Investigação da composição do EBITDA

A diferença entre o EBITDA informado e o EBITDA reconstruído ainda não foi explicada.

Nesta etapa, vamos comparar essa diferença com todas as contas da DRE para identificar se alguma delas corresponde ao ajuste utilizado no cálculo original.

In [ ]:
diferenca_ebitda = (
    dre_organizada.loc["DRE - EBITDA"]
    - (
        dre_organizada.loc["DRE - Resultado Operacional"]
        - dre_organizada.loc["DRE - Depreciação e Amortização"]
        + dre_organizada.loc["DRE - Outros Resultados Operacionais"]
    )
)

# Todas as contas da DRE do cenário
investigacao_ebitda = dre_organizada.copy()

# Adiciona a diferença como uma linha de referência
investigacao_ebitda.loc["AJUSTE NECESSÁRIO PARA EBITDA"] = diferenca_ebitda

print("Investigação da composição do EBITDA —", cenario_selecionado)

display(investigacao_ebitda)

## Margem EBITDA

A Margem EBITDA mostra quanto do faturamento da empresa corresponde ao EBITDA.

O cálculo será feito dividindo o EBITDA pela Receita.

In [ ]:
# Contas utilizadas

ebitda = dre_organizada.loc[
    "DRE - EBITDA"
]

receita = dre_organizada.loc[
    "DRE - Receita"
]

# Cálculo da Margem EBITDA

margem_ebitda = ebitda / receita

# Tabela do indicador

margem_ebitda_df = pd.DataFrame({
    "EBITDA": ebitda,
    "Receita": receita,
    "Margem EBITDA": margem_ebitda
})

print("Margem EBITDA —", cenario_selecionado)

display(margem_ebitda_df)

## Verificação dos componentes do EVA

O EVA mede a geração de valor econômico da empresa.

Para calculá-lo, precisamos identificar três componentes:
- NOPAT
- Capital Investido
- WACC

Antes do cálculo, vamos verificar quais informações estão disponíveis na base tratada.

In [ ]:
# Procurar contas relacionadas aos componentes do EVA

contas_eva = [
    conta for conta in df_kpis["Conta"].unique()
    if any(
        termo in conta.lower()
        for termo in [
            "nopat",
            "capital investido",
            "wacc",
            "custo de capital"
        ]
    )
]

print("Contas relacionadas ao EVA encontradas na base:")

if contas_eva:
    for conta in contas_eva:
        print("-", conta)
else:
    print("Nenhuma conta encontrada.")

## Verificação dos dados necessários para o EVA

A base não possui contas explícitas de NOPAT, Capital Investido ou WACC.

Por isso, vamos verificar se existem contas que permitam derivar esses componentes a partir das demonstrações contábeis.

In [ ]:
# Contas que podem ser utilizadas na análise do EVA

contas_eva_base = [
    "DRE - Resultado Operacional",
    "DRE - Resultado Antes do Imposto de Renda",
    "DRE - Imposto de Renda e Contribuição Social",
    "DRE - EBITDA",
    "BAL - Total do Ativo",
    "BAL - Disponível",
    "BAL - Patrimônio Líquido",
    "BAL - Emprést",
    "BAL - Empréstimos",
    "BAL - Exigível a Longo Prazo",
    "BAL - Passivo Circulante"
]

verificacao_eva = pd.DataFrame(
    index=contas_eva_base,
    columns=ordem_anos
)

for conta in contas_eva_base:
    if conta in consulta.index:
        verificacao_eva.loc[conta] = consulta.loc[conta]
    else:
        verificacao_eva.loc[conta] = None

print("Dados disponíveis para análise do EVA —", cenario_selecionado)

display(verificacao_eva)

## Investigação da taxa implícita da dívida

A base não apresenta uma taxa de juros de forma explícita.

Como existem despesas financeiras e contas de empréstimos, vamos investigar se é possível identificar uma taxa implícita de custo da dívida.

Essa análise é exploratória e não significa, ainda, que a taxa encontrada seja o WACC.

In [ ]:
# Valores financeiros utilizados na investigação

despesas_financeiras = dre_organizada.loc[
    "DRE - Despesas Financeiras"
].abs()

emprestimo = bal_organizado.loc[
    "BAL - Emprést"
].abs()

emprestimos = bal_organizado.loc[
    "BAL - Empréstimos"
].abs()

exigivel_lp = bal_organizado.loc[
    "BAL - Exigível a Longo Prazo"
].abs()

passivo_circulante = bal_organizado.loc[
    "BAL - Passivo Circulante"
].abs()

# Dívida identificada diretamente pelos empréstimos

divida_emprestimos = emprestimo + emprestimos

# Taxa implícita simples

taxa_emprestimos = despesas_financeiras / divida_emprestimos

# Tabela

taxa_divida = pd.DataFrame({
    "Despesas Financeiras": despesas_financeiras,
    "Empréstimo": emprestimo,
    "Empréstimos": emprestimos,
    "Dívida por Empréstimos": divida_emprestimos,
    "Taxa Implícita": taxa_emprestimos
})

print("Investigação da taxa implícita da dívida —", cenario_selecionado)

display(taxa_divida)

## Investigação da taxa da dívida pela dívida média

A taxa anterior foi calculada usando o saldo da dívida no final de cada ano.

Agora vamos testar uma segunda aproximação, usando a dívida média entre o ano atual e o ano anterior. Isso ajuda a verificar se as despesas financeiras apresentam uma relação mais consistente com o nível de endividamento.

Essa taxa continua sendo apenas uma estimativa exploratória do custo da dívida e não deve ser tratada como WACC.

In [ ]:
# Dívida total identificada pelas contas de empréstimos

divida = (
    bal_organizado.loc["BAL - Emprést"].abs()
    + bal_organizado.loc["BAL - Empréstimos"].abs()
)

# Dívida média entre o ano anterior e o ano atual
divida_media = (divida + divida.shift(1)) / 2

# Taxa implícita usando a dívida média
taxa_divida_media = despesas_financeiras / divida_media

taxa_divida_media_df = pd.DataFrame({
    "Despesas Financeiras": despesas_financeiras,
    "Dívida Final": divida,
    "Dívida Média": divida_media,
    "Taxa Implícita pela Dívida Média": taxa_divida_media
})

print("Taxa implícita pela dívida média —", cenario_selecionado)

display(taxa_divida_media_df)

## Verificação dos componentes do EVA

O EVA depende de três informações principais:

- NOPAT
- Capital Investido
- WACC

Já verificamos que a base não apresenta o WACC de forma explícita.

Agora vamos verificar se NOPAT e Capital Investido podem ser obtidos a partir das contas disponíveis, sem criar premissas arbitrárias.

In [ ]:
# Contas disponíveis que podem participar da análise do EVA

contas_eva = [
    "DRE - Resultado Operacional",
    "DRE - Imposto de Renda e Contribuição Social",
    "BAL - Ativo Circulante",
    "BAL - Realizável a Longo Prazo",
    "BAL - Permanente",
    "BAL - Total do Ativo",
    "BAL - Passivo Circulante",
    "BAL - Exigível a Longo Prazo",
    "BAL - Patrimônio Líquido"
]

componentes_eva = bal_organizado.reindex(
    [conta for conta in contas_eva if conta in bal_organizado.index]
)

# Adiciona as contas da DRE que existem no demonstrativo
componentes_eva_dre = dre_organizada.reindex(
    [conta for conta in contas_eva if conta in dre_organizada.index]
)

print("Componentes disponíveis para investigação do EVA —", cenario_selecionado)

print("\nContas do Balanço:")
display(componentes_eva)

print("\nContas da DRE:")
display(componentes_eva_dre)

## Investigação do Capital Investido

O Capital Investido não está apresentado diretamente na base.

Por isso, vamos verificar como os principais componentes do Balanço se relacionam entre si.

Nesta etapa, os cálculos são apenas exploratórios. O objetivo é identificar uma composição que seja consistente com a estrutura da base antes de utilizá-la no EVA.

In [ ]:
# Componentes do Balanço utilizados na investigação

ativo_circulante = bal_organizado.loc[
    "BAL - Ativo Circulante"
]

realizavel_lp = bal_organizado.loc[
    "BAL - Realizável a Longo Prazo"
]

permanente = bal_organizado.loc[
    "BAL - Permanente"
]

passivo_circulante = bal_organizado.loc[
    "BAL - Passivo Circulante"
]

exigivel_lp = bal_organizado.loc[
    "BAL - Exigível a Longo Prazo"
]

patrimonio_liquido = bal_organizado.loc[
    "BAL - Patrimônio Líquido"
]

# Ativo operacional / total de ativos
capital_ativo = (
    ativo_circulante
    + realizavel_lp
    + permanente
)

# Capital de terceiros
capital_terceiros = (
    passivo_circulante.abs()
    + exigivel_lp.abs()
)

# Estrutura de capital pela soma dos valores absolutos
capital_total = (
    capital_terceiros
    + patrimonio_liquido.abs()
)

investigacao_capital = pd.DataFrame({
    "Ativo Total": capital_ativo,
    "Capital de Terceiros": capital_terceiros,
    "Patrimônio Líquido": patrimonio_liquido.abs(),
    "Capital Total": capital_total
})

print("Investigação do Capital Investido —", cenario_selecionado)

display(investigacao_capital)

## Margem Líquida

A Margem Líquida mostra quanto da receita permanece como resultado líquido após os efeitos das demais despesas e impostos.

In [ ]:
resultado_liquido = dre_organizada.loc[
    "DRE - Resultado Líquido"
]

receita = dre_organizada.loc[
    "DRE - Receita"
]

margem_liquida = resultado_liquido / receita

margem_liquida_df = pd.DataFrame({
    "Resultado Líquido": resultado_liquido,
    "Receita": receita,
    "Margem Líquida": margem_liquida
})

print("Margem Líquida —", cenario_selecionado)

display(margem_liquida_df)

## Consolidação dos indicadores financeiros

Nesta etapa, reunimos os indicadores calculados em uma única base.

A tabela final será utilizada posteriormente para comparar os cenários e realizar o ranking.

In [5]:
# Consolidando os indicadores calculados para o cenário selecionado

kpis_cenario = pd.DataFrame({
    "Cenario": cenario_selecionado,
    "Ano": ordem_anos,
    "Receita": receita.values,
    "Custos": custos.values,
    "Resultado Líquido": resultado_liquido.values,
    "EBITDA": ebitda.values,
    "Margem EBITDA": margem_ebitda.values,
    "Margem Líquida": margem_liquida.values,
    "Liquidez Corrente": lc.values,
    "Liquidez Seca": ls.values,
    "Liquidez Imediata": li.values,
    "Liquidez Geral": lg.values,
    "PCT": pct.values,
    "CE": ce.values,
    "IPL": ipl.values
})

print("Tabela consolidada de KPIs —", cenario_selecionado)

display(kpis_cenario)

NameError: name 'cenario_selecionado' is not defined

## Consolidação dos KPIs

Nesta etapa, os indicadores calculados anteriormente serão aplicados a todos os cenários e anos disponíveis na base.

O resultado será uma tabela consolidada com 1.200 cenários e 12 anos, totalizando 14.400 observações.

In [3]:

import numpy as np
import pandas as pd

wacc = 0.08
ordem_anos = [f"Ano {i}" for i in range(1, 13)]
resultados = []


def dividir_seguro(numerador, denominador):
    resultado = numerador.div(denominador)
    return resultado.replace([np.inf, -np.inf], np.nan)


for cenario, grupo in df_kpis.groupby("Cenario", sort=False):

    tabela = (
        grupo.pivot_table(
            index="Conta",
            columns="Ano",
            values="Valor_numerico",
            aggfunc="first"
        )
        .reindex(columns=ordem_anos)
    )

    def conta(nome):
        if nome in tabela.index:
            return pd.to_numeric(tabela.loc[nome], errors="coerce")
        return pd.Series(np.nan, index=ordem_anos, dtype="float64")

    receita = conta("DRE - Receita")
    tributos = conta("DRE - Tributos")
    custos = conta("DRE - Custos")
    ebitda = conta("DRE - EBITDA")
    resultado_operacional = conta("DRE - Resultado Operacional")
    resultado_antes_ir = conta("DRE - Resultado Antes do Imposto de Renda")
    imposto_ir = conta("DRE - Imposto de Renda e Contribuição Social")
    resultado_liquido = conta("DRE - Resultado Líquido")
    deprec_amort = conta("DRE - Depreciação e Amortização")

    ativo = conta("BAL - Total do Ativo").abs()
    ativo_circulante = conta("BAL - Ativo Circulante")
    disponivel = conta("BAL - Disponível").abs()
    estoques = conta("BAL - Estoques Diversos")
    realizavel_lp = conta("BAL - Realizável a Longo Prazo")
    passivo_circulante = conta("BAL - Passivo Circulante").abs()
    exigivel_lp = conta("BAL - Exigível a Longo Prazo").abs()
    patrimonio_liquido = conta("BAL - Patrimônio Líquido").abs()
    permanente = conta("BAL - Permanente")

    emprestimo_cp = conta("BAL - Emprést").abs()
    emprestimos_lp = conta("BAL - Empréstimos").abs()

    clientes = conta("BAL - Contas a Receber - Clientes").abs()
    fornecedores = conta("BAL - Fornecedores").abs()

    recebimentos_flu = conta("FLU - Receita")
    tributos_pagos = conta("FLU - Tributos")
    custos_pagos = conta("FLU - Custos")
    ir_pago = conta("FLU - Imposto de Renda e Contribuição Social")
    investimentos = conta("FLU - Investimentos").abs()
    juros_pagos = conta("FLU - Despesas Financeiras").abs()
    distribuicao_acionistas = conta("FLU - Distribuição para Acionista").abs()

    receita_liquida = receita + tributos

    margem_ebitda = dividir_seguro(ebitda, receita_liquida)
    margem_liquida = dividir_seguro(resultado_liquido, receita_liquida)

    lc = dividir_seguro(ativo_circulante, passivo_circulante)
    ls = dividir_seguro(
        ativo_circulante - estoques,
        passivo_circulante
    )
    li = dividir_seguro(disponivel, passivo_circulante)
    lg = dividir_seguro(
        ativo_circulante + realizavel_lp,
        passivo_circulante + exigivel_lp
    )

    capital_terceiros = passivo_circulante + exigivel_lp

    pct = dividir_seguro(capital_terceiros, patrimonio_liquido)
    ce = dividir_seguro(passivo_circulante, capital_terceiros)
    ipl = dividir_seguro(permanente, patrimonio_liquido)

    aliquota_ir_cs = dividir_seguro(
        imposto_ir.abs(),
        resultado_antes_ir
    )

    nopat = resultado_operacional * (1 - aliquota_ir_cs)

    capital_investido = (
        patrimonio_liquido
        + emprestimo_cp
        + emprestimos_lp
        - disponivel
    )

    roic = dividir_seguro(nopat, capital_investido)
    eva = nopat - (capital_investido * wacc)
    mva_aproximado = dividir_seguro(eva, pd.Series(wacc, index=ordem_anos))

    ativo_medio = (ativo + ativo.shift(1)) / 2
    pl_medio = (patrimonio_liquido + patrimonio_liquido.shift(1)) / 2

    roe = dividir_seguro(resultado_liquido, pl_medio)

    dupont_margem = dividir_seguro(resultado_liquido, receita_liquida)
    dupont_giro = dividir_seguro(receita_liquida, ativo_medio)
    dupont_alavancagem = dividir_seguro(ativo_medio, pl_medio)

    roe_dupont = (
        dupont_margem
        * dupont_giro
        * dupont_alavancagem
    )

    ccl = ativo_circulante - passivo_circulante

    pmr = dividir_seguro(clientes, receita) * 360
    pmp = dividir_seguro(fornecedores, custos.abs()) * 360
    ciclo_financeiro = pmr - pmp

    necessidade_capital_giro = (
        ciclo_financeiro * custos.abs() / 360
    )

    variacao_clientes = clientes.diff()

    recebimentos_estimados = receita - variacao_clientes

    fco_estimado = (
        recebimentos_estimados
        + tributos_pagos
        + custos_pagos
        + ir_pago
    )

    taxa_conversao_caixa = dividir_seguro(
        recebimentos_estimados,
        receita
    )

    fco_ebitda = dividir_seguro(fco_estimado, ebitda)
    fco_lucro = dividir_seguro(fco_estimado, resultado_liquido)

    fcl = fco_estimado - investimentos

    capex_receita = dividir_seguro(investimentos, receita)
    capex_depreciacao = dividir_seguro(investimentos, deprec_amort.abs())

    cobertura_juros = dividir_seguro(fco_estimado, juros_pagos)
    fco_divida = dividir_seguro(
        fco_estimado,
        emprestimo_cp + emprestimos_lp
    )

    payout_caixa = dividir_seguro(
        distribuicao_acionistas,
        resultado_liquido
    )

    cobertura_dividendos = dividir_seguro(
        fcl,
        distribuicao_acionistas
    )

    for i, ano in enumerate(ordem_anos):

        resultados.append({
            "Cenario": cenario,
            "Ano": ano,

            "Receita": receita.loc[ano],
            "Receita Líquida": receita_liquida.loc[ano],
            "Custos": custos.loc[ano],
            "EBITDA": ebitda.loc[ano],
            "Resultado Líquido": resultado_liquido.loc[ano],

            "Margem EBITDA": margem_ebitda.loc[ano],
            "Margem Líquida": margem_liquida.loc[ano],

            "LC": lc.loc[ano],
            "LS": ls.loc[ano],
            "LI": li.loc[ano],
            "LG": lg.loc[ano],

            "PCT": pct.loc[ano],
            "CE": ce.loc[ano],
            "IPL": ipl.loc[ano],

            "NOPAT": nopat.loc[ano],
            "Capital Investido": capital_investido.loc[ano],
            "ROIC": roic.loc[ano],
            "WACC": wacc,
            "EVA": eva.loc[ano],
            "MVA Aproximado": mva_aproximado.loc[ano],

            "ROE": roe.loc[ano],
            "DuPont - Margem Líquida": dupont_margem.loc[ano],
            "DuPont - Giro do Ativo": dupont_giro.loc[ano],
            "DuPont - Alavancagem": dupont_alavancagem.loc[ano],
            "ROE DuPont": roe_dupont.loc[ano],

            "CCL": ccl.loc[ano],
            "PMR": pmr.loc[ano],
            "PMP": pmp.loc[ano],
            "Ciclo Financeiro": ciclo_financeiro.loc[ano],
            "Necessidade de Capital de Giro": necessidade_capital_giro.loc[ano],

            "Recebimentos Estimados": recebimentos_estimados.loc[ano],
            "Taxa de Conversão da Receita em Caixa": taxa_conversao_caixa.loc[ano],
            "FCO": fco_estimado.loc[ano],
            "FCO / EBITDA": fco_ebitda.loc[ano],
            "FCO / Lucro Líquido": fco_lucro.loc[ano],
            "FCL": fcl.loc[ano],
            "Capex / Receita": capex_receita.loc[ano],
            "Capex / D&A": capex_depreciacao.loc[ano],
            "Cobertura de Juros": cobertura_juros.loc[ano],
            "FCO / Dívida Bruta": fco_divida.loc[ano],
            "Payout em Caixa": payout_caixa.loc[ano],
            "Cobertura de Dividendos": cobertura_dividendos.loc[ano]
        })

kpis_final = pd.DataFrame(resultados)

print("Quantidade de cenários:", kpis_final["Cenario"].nunique())
print("Quantidade de anos:", kpis_final["Ano"].nunique())
print("Quantidade de registros:", len(kpis_final))

display(kpis_final.head(12))


kpis_novos = kpis_final.copy()


Quantidade de cenários: 1200
Quantidade de anos: 12
Quantidade de registros: 14400


,Cenario,Ano,Receita,Receita Líquida,Custos,EBITDA,Resultado Líquido,Margem EBITDA,Margem Líquida,LC,...,FCO,FCO / EBITDA,FCO / Lucro Líquido,FCL,Capex / Receita,Capex / D&A,Cobertura de Juros,FCO / Dívida Bruta,Payout em Caixa,Cobertura de Dividendos
0,Total Cen_00001,Ano 1,6.722729e+09,6.251265e+09,-1.361165e+09,5.188381e+09,2.443243e+09,0.829973,0.390840,1.661277,...,NaN,NaN,NaN,NaN,0.079432,0.884736,NaN,NaN,0.627202,NaN
1,Total Cen_00001,Ano 2,6.114937e+09,5.611118e+09,-1.333291e+09,4.630952e+09,2.000834e+09,0.825317,0.356584,1.154874,...,3.539101e+09,0.764228,1.768813,3.207066e+09,0.054299,0.455924,5.075475,0.592057,1.276341,1.255828
2,Total Cen_00001,Ano 3,6.251283e+09,5.721581e+09,-1.232953e+09,4.883290e+09,2.123146e+09,0.853486,0.371077,1.301103,...,3.560205e+09,0.729059,1.676854,3.412020e+09,0.023705,0.190287,5.948743,0.591540,0.974845,1.648528
3,Total Cen_00001,Ano 4,6.506532e+09,5.949170e+09,-1.236163e+09,5.148066e+09,2.318127e+09,0.865342,0.389656,3.324041,...,3.319987e+09,0.644900,1.432185,3.233545e+09,0.013285,0.106199,1.878050,0.511708,0.957944,1.456134
4,Total Cen_00001,Ano 5,6.844711e+09,6.255466e+09,-1.127881e+09,5.453240e+09,2.642606e+09,0.871756,0.422448,0.664478,...,3.253466e+09,0.596612,1.231158,3.208787e+09,0.006527,0.053871,1.979204,0.423107,0.931011,1.304228
5,Total Cen_00001,Ano 6,7.270773e+09,6.644719e+09,-1.140222e+09,5.834764e+09,2.955369e+09,0.878105,0.444770,0.395148,...,3.955358e+09,0.677895,1.338363,3.865369e+09,0.012377,0.106196,3.455583,0.462726,0.934086,1.400207
6,Total Cen_00001,Ano 7,7.720096e+09,7.053177e+09,-1.269192e+09,6.236914e+09,3.153307e+09,0.884270,0.447076,0.379970,...,4.248933e+09,0.681256,1.347453,4.196446e+09,0.006799,0.060292,3.999756,0.451680,0.948256,1.403427
7,Total Cen_00001,Ano 8,8.248988e+09,7.536001e+09,-1.342629e+09,6.693782e+09,3.428037e+09,0.888241,0.454888,0.764404,...,4.518022e+09,0.674958,1.317962,4.483690e+09,0.004162,0.038598,11.928195,0.464705,0.933650,1.400896
8,Total Cen_00001,Ano 9,8.756416e+09,8.000228e+09,-1.412146e+09,7.136917e+09,3.734760e+09,0.892089,0.466832,0.809218,...,4.376835e+09,0.613267,1.171919,4.235538e+09,0.016136,0.156229,12.228714,0.435150,0.924226,1.227065
9,Total Cen_00001,Ano 10,9.334636e+09,8.529061e+09,-1.277249e+09,7.653142e+09,4.238241e+09,0.897302,0.496918,0.646628,...,4.413762e+09,0.576725,1.041414,4.359659e+09,0.005796,0.058008,13.114676,0.423715,0.896271,1.147698


## Validação dos indicadores

A tabela abaixo apresenta uma visão resumida da validação da base e dos indicadores calculados.



In [ ]:
# Validação estrutural da tabela final

validacao = pd.DataFrame({
    "Indicador": [
        "Receita",
        "Custos",
        "EBITDA",
        "Margem EBITDA",
        "Resultado Líquido",
        "Margem Líquida",
        "Liquidez Corrente",
        "Liquidez Seca",
        "Liquidez Imediata",
        "Liquidez Geral",
        "PCT",
        "CE",
        "IPL"
    ],
    "Status": [
        "OK",
        "OK",
        "OK",
        "OK",
        "OK",
        "OK",
        "OK",
        "OK*",
        "OK",
        "OK*",
        "OK*",
        "OK*",
        "OK*"
    ],
    "Observação": [
        "Disponível na DRE",
        "Disponível como custo total",
        "Fornecido pela base",
        "Calculada sobre a receita",
        "Disponível na DRE",
        "Calculada sobre a receita",
        "Calculada com dados do BP",
        "Ano 12 possui ausência estrutural de estoque",
        "Calculada com o disponível",
        "Ano 12 possui ausência estrutural no ELP",
        "Calculado com capital de terceiros",
        "Calculado com passivo circulante",
        "Calculado com permanente e PL"
    ]
})

display(validacao)

## Índices de Endividamento — Cenário 1

Os índices de endividamento permitem analisar a participação do capital de terceiros na estrutura financeira da empresa.

Para o cenário selecionado, serão calculados:

- Participação de Capital de Terceiros (PCT);
- Composição do Endividamento (CE);
- Imobilização do Patrimônio Líquido (IPL).

Os cálculos serão realizados para os 12 anos disponíveis na base.

## Indicadores de Geração de Valor

Nesta etapa, são preparados os dados necessários para o cálculo dos indicadores de geração de valor.

In [ ]:
# Valores do balanço utilizados nos cálculos
passivo_circulante = bal_organizado.loc[
    "BAL - Passivo Circulante"
].abs()

exigivel_lp = bal_organizado.loc[
    "BAL - Exigível a Longo Prazo"
].abs()

patrimonio_liquido = bal_organizado.loc[
    "BAL - Patrimônio Líquido"
].abs()

permanente = bal_organizado.loc[
    "BAL - Permanente"
]

# Capital de terceiros = dívidas de curto + longo prazo
capital_terceiros = passivo_circulante + exigivel_lp

# Índices de endividamento
pct = capital_terceiros / patrimonio_liquido

ce = passivo_circulante / capital_terceiros

ipl = permanente / patrimonio_liquido

# Tabela final
indices_endividamento = pd.DataFrame({
    "Passivo Circulante": passivo_circulante,
    "Exigível a Longo Prazo": exigivel_lp,
    "Patrimônio Líquido": patrimonio_liquido,
    "Capital de Terceiros": capital_terceiros,
    "PCT": pct,
    "CE": ce,
    "IPL": ipl
})

display(indices_endividamento)

### Componentes para o cálculo do EVA

São identificadas as contas necessárias para o cálculo e a análise do EVA.

In [ ]:
# Componentes disponíveis para análise do EVA

nopat_conta = "DRE - Resultado Operacional"
imposto_conta = "DRE - Imposto de Renda e Contribuição Social"

contas_capital = [
    "BAL - Ativo Circulante",
    "BAL - Realizável a Longo Prazo",
    "BAL - Permanente",
    "BAL - Passivo Circulante",
    "BAL - Exigível a Longo Prazo",
    "BAL - Patrimônio Líquido"
]

# Verifica quais contas existem na base organizada
componentes_eva = pd.DataFrame(index=ordem_anos)

componentes_eva["Resultado Operacional"] = dre_organizada.loc[
    nopat_conta
].values

componentes_eva["IR e CS"] = dre_organizada.loc[
    imposto_conta
].values

for conta in contas_capital:
    componentes_eva[conta.replace("BAL - ", "")] = bal_organizado.loc[
        conta
    ].values

display(componentes_eva)

### Cálculo do NOPAT

O NOPAT representa o resultado operacional após o efeito estimado dos impostos.

In [ ]:
# Resultado operacional utilizado como referência para o EBIT
ebit = dre_organizada.loc[
    "DRE - Resultado Operacional"
]

# Resultado antes do IR e CS
resultado_antes_ir = dre_organizada.loc[
    "DRE - Resultado Antes do Imposto de Renda"
]

# Imposto de Renda e Contribuição Social
ir_cs = dre_organizada.loc[
    "DRE - Imposto de Renda e Contribuição Social"
]

# Alíquota efetiva de IR e CS
aliquota_ir_cs = ir_cs.abs() / resultado_antes_ir

# Imposto estimado sobre o resultado operacional
imposto_sobre_ebit = ebit * aliquota_ir_cs

# NOPAT
nopat = ebit - imposto_sobre_ebit

nopat_df = pd.DataFrame({
    "EBIT / Resultado Operacional": ebit,
    "Resultado Antes do IR e CS": resultado_antes_ir,
    "IR e CS": ir_cs,
    "Alíquota Efetiva": aliquota_ir_cs,
    "Imposto sobre EBIT": imposto_sobre_ebit,
    "NOPAT": nopat
})

display(nopat_df)

### Capital Investido

O Capital Investido considera os recursos utilizados pela empresa como base para a análise de geração de valor.

In [ ]:
# Capital Investido conforme a metodologia apresentada pelo professor

patrimonio_liquido = bal_organizado.loc[
    "BAL - Patrimônio Líquido"
].abs()

emprestimo = bal_organizado.loc[
    "BAL - Emprést"
].abs()

emprestimos = bal_organizado.loc[
    "BAL - Empréstimos"
].abs()

divida_financeira = emprestimo + emprestimos

capital_investido = (
    patrimonio_liquido
    + divida_financeira
)

capital_investido_df = pd.DataFrame({
    "Patrimônio Líquido": patrimonio_liquido,
    "Emprést": emprestimo,
    "Empréstimos": emprestimos,
    "Dívida Financeira": divida_financeira,
    "Capital Investido": capital_investido
})

display(capital_investido_df)

### Cálculo do ROIC

O ROIC relaciona o NOPAT ao Capital Investido, indicando o retorno obtido sobre o capital utilizado.

In [ ]:
roic = nopat / capital_investido

roic_df = pd.DataFrame({
    "NOPAT": nopat,
    "Capital Investido": capital_investido,
    "ROIC": roic
})

display(roic_df)

### Cálculo do EVA

Para o cálculo do EVA, foi adotado um WACC de 8%, permitindo avaliar a geração de valor econômico.

In [ ]:
disponivel = bal_organizado.loc[
    "BAL - Disponível"
].abs()

capital_investido_eva = (
    patrimonio_liquido
    + emprestimo
    + emprestimos
    - disponivel
)

wacc = 0.08

encargo_capital = capital_investido_eva * wacc

eva = nopat - encargo_capital

eva_df = pd.DataFrame({
    "NOPAT": nopat,
    "Capital Investido": capital_investido_eva,
    "WACC": wacc,
    "Encargo de Capital": encargo_capital,
    "EVA": eva
})

display(eva_df)

### MVA Aproximado

O MVA Aproximado utiliza o EVA e o WACC como uma estimativa complementar da geração de valor.

In [ ]:
mva_aproximado = eva / wacc

mva_df = pd.DataFrame({
    "EVA": eva,
    "WACC": wacc,
    "MVA Aproximado": mva_aproximado
})

display(mva_df)

## Consolidação dos Indicadores

Após a validação dos cálculos, os indicadores são aplicados aos 1.200 cenários e aos 12 anos disponíveis na base.

In [4]:
# Consolidação dos KPIs para todos os cenários e anos

wacc = 0.08
resultados = []

ordem_anos = [f"Ano {i}" for i in range(1, 13)]

for cenario, grupo in df_kpis.groupby("Cenario"):

    tabela = (
        grupo
        .pivot_table(
            index="Conta",
            columns="Ano",
            values="Valor_numerico",
            aggfunc="first"
        )
        .reindex(columns=ordem_anos)
    )

    def conta(nome):
        if nome in tabela.index:
            return tabela.loc[nome]
        return pd.Series(index=ordem_anos, dtype=float)

    # DRE
    receita = conta("DRE - Receita")
    tributos = conta("DRE - Tributos")
    ebitda = conta("DRE - EBITDA")
    resultado_operacional = conta("DRE - Resultado Operacional")
    resultado_antes_ir = conta("DRE - Resultado Antes do Imposto de Renda")
    imposto_ir = conta("DRE - Imposto de Renda e Contribuição Social")
    resultado_liquido = conta("DRE - Resultado Líquido")

    # BAL
    ativo_circulante = conta("BAL - Ativo Circulante")
    disponivel = conta("BAL - Disponível")
    estoques = conta("BAL - Estoques Diversos")
    realizavel_lp = conta("BAL - Realizável a Longo Prazo")
    passivo_circulante = conta("BAL - Passivo Circulante").abs()
    exigivel_lp = conta("BAL - Exigível a Longo Prazo").abs()
    patrimonio_liquido = conta("BAL - Patrimônio Líquido").abs()
    permanente = conta("BAL - Permanente")
    emprestimo = conta("BAL - Emprést").abs()
    emprestimos = conta("BAL - Empréstimos").abs()

    # Margens
    receita_liquida = receita + tributos

    margem_ebitda = ebitda / receita_liquida
    margem_liquida = resultado_liquido / receita_liquida

    # Liquidez
    lc = ativo_circulante / passivo_circulante
    ls = (ativo_circulante - estoques) / passivo_circulante
    li = disponivel / passivo_circulante
    lg = (
        (ativo_circulante + realizavel_lp)
        / (passivo_circulante + exigivel_lp)
    )

    # Endividamento
    capital_terceiros = passivo_circulante + exigivel_lp

    pct = capital_terceiros / patrimonio_liquido
    ce = passivo_circulante / capital_terceiros
    ipl = permanente / patrimonio_liquido

    # NOPAT
    aliquota_ir_cs = imposto_ir.abs() / resultado_antes_ir
    imposto_sobre_ebit = resultado_operacional * aliquota_ir_cs
    nopat = resultado_operacional - imposto_sobre_ebit

    # Capital Investido
    # Metodologia adotada anteriormente:
    # Patrimônio Líquido + dívidas financeiras - Disponível
    capital_investido = (
        patrimonio_liquido
        + emprestimo
        + emprestimos
        - disponivel
    )

    # ROIC
    roic = nopat / capital_investido

    # EVA
    encargo_capital = capital_investido * wacc
    eva = nopat - encargo_capital

    # MVA aproximado pela perpetuidade do EVA
    mva_aproximado = eva / wacc

    # Consolidação
    for ano in ordem_anos:
        resultados.append({
            "Cenario": cenario,
            "Ano": ano,
            "Margem EBITDA": margem_ebitda[ano],
            "Margem Líquida": margem_liquida[ano],
            "LC": lc[ano],
            "LS": ls[ano],
            "LI": li[ano],
            "LG": lg[ano],
            "PCT": pct[ano],
            "CE": ce[ano],
            "IPL": ipl[ano],
            "NOPAT": nopat[ano],
            "Capital Investido": capital_investido[ano],
            "ROIC": roic[ano],
            "WACC": wacc,
            "EVA": eva[ano],
            "MVA Aproximado": mva_aproximado[ano]
        })

kpis_final = pd.DataFrame(resultados)

if "kpis_novos" not in globals():
    raise NameError(
        "A tabela 'kpis_novos' não existe. Execute antes a célula da seção "
        "'Consolidação dos KPIs' (a que calcula ROE, DuPont, FCO, FCL etc.)."
    )

colunas_novos_indicadores = [
    "Recebimentos Estimados",
    "ROE",
    "DuPont - Margem Líquida",
    "DuPont - Giro do Ativo",
    "DuPont - Alavancagem",
    "ROE DuPont",
    "CCL",
    "PMR",
    "PMP",
    "Ciclo Financeiro",
    "Necessidade de Capital de Giro",
    "Taxa de Conversão da Receita em Caixa",
    "FCO",
    "FCO / EBITDA",
    "FCO / Lucro Líquido",
    "FCL",
    "Capex / Receita",
    "Capex / D&A",
    "Cobertura de Juros",
    "FCO / Dívida Bruta",
    "Payout em Caixa",
    "Cobertura de Dividendos",
]

kpis_final = kpis_final.merge(
    kpis_novos[["Cenario", "Ano"] + colunas_novos_indicadores],
    on=["Cenario", "Ano"],
    how="left",
    validate="one_to_one"
)

print("Quantidade de cenários:", kpis_final["Cenario"].nunique())
print("Quantidade de anos:", kpis_final["Ano"].nunique())
print("Quantidade de registros:", len(kpis_final))

display(kpis_final.head(12))

Quantidade de cenários: 1200
Quantidade de anos: 12
Quantidade de registros: 14400


,Cenario,Ano,Margem EBITDA,Margem Líquida,LC,LS,LI,LG,PCT,CE,...,FCO,FCO / EBITDA,FCO / Lucro Líquido,FCL,Capex / Receita,Capex / D&A,Cobertura de Juros,FCO / Dívida Bruta,Payout em Caixa,Cobertura de Dividendos
0,Total Cen_00001,Ano 1,0.829973,0.390840,1.661277,1.661197,1.115028,0.282561,4.277473,0.125446,...,NaN,NaN,NaN,NaN,0.079432,0.884736,NaN,NaN,0.627202,NaN
1,Total Cen_00001,Ano 2,0.825317,0.356584,1.154874,1.154811,0.753056,0.278254,6.085068,0.152663,...,3.539101e+09,0.764228,1.768813,3.207066e+09,0.054299,0.455924,5.075475,0.592057,1.276341,1.255828
2,Total Cen_00001,Ano 3,0.853486,0.371077,1.301103,1.301054,0.992276,0.376489,6.137439,0.189511,...,3.560205e+09,0.729059,1.676854,3.412020e+09,0.023705,0.190287,5.948743,0.591540,0.974845,1.648528
3,Total Cen_00001,Ano 4,0.865342,0.389656,3.324041,3.323867,2.243316,0.376906,4.982482,0.062000,...,3.319987e+09,0.644900,1.432185,3.233545e+09,0.013285,0.106199,1.878050,0.511708,0.957944,1.456134
4,Total Cen_00001,Ano 5,0.871756,0.422448,0.664478,0.664409,0.243336,0.250630,4.902671,0.143564,...,3.253466e+09,0.596612,1.231158,3.208787e+09,0.006527,0.053871,1.979204,0.423107,0.931011,1.304228
5,Total Cen_00001,Ano 6,0.878105,0.444770,0.395148,0.395105,0.133008,0.251326,4.793570,0.209836,...,3.955358e+09,0.677895,1.338363,3.865369e+09,0.012377,0.106196,3.455583,0.462726,0.934086,1.400207
6,Total Cen_00001,Ano 7,0.884270,0.447076,0.379970,0.379935,0.172747,0.288164,4.650721,0.247296,...,4.248933e+09,0.681256,1.347453,4.196446e+09,0.006799,0.060292,3.999756,0.451680,0.948256,1.403427
7,Total Cen_00001,Ano 8,0.888241,0.454888,0.764404,0.764366,0.588516,0.415432,4.142171,0.236112,...,4.518022e+09,0.674958,1.317962,4.483690e+09,0.004162,0.038598,11.928195,0.464705,0.933650,1.400896
8,Total Cen_00001,Ano 9,0.892089,0.466832,0.809218,0.809186,0.704683,0.458381,3.850995,0.266129,...,4.376835e+09,0.613267,1.171919,4.235538e+09,0.016136,0.156229,12.228714,0.435150,0.924226,1.227065
9,Total Cen_00001,Ano 10,0.897302,0.496918,0.646628,0.646606,0.610827,0.431964,3.651045,0.336053,...,4.413762e+09,0.576725,1.041414,4.359659e+09,0.005796,0.058008,13.114676,0.423715,0.896271,1.147698


In [ ]:
# Exportação PRELIMINAR (sem Ranking e sem Análise Comparativa).
# Esta célula é opcional: a planilha final é gerada na célula de exportação ao fim do notebook.

from pathlib import Path
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.utils import get_column_letter
from openpyxl.workbook.defined_name import DefinedName


# Entrega 2: a planilha é gerada na pasta da Entrega 2 (a Entrega 1 não é alterada).
# Se a pasta ainda não existir, ela é criada.
pasta_saida = Path("../documentos/entrega 2/contabilidade e finanças")
pasta_saida.mkdir(parents=True, exist_ok=True)

arquivo_saida = pasta_saida / "Analise_Financeira.xlsx"


base = df_kpis[
    ["Cenario", "Ano", "Conta", "Valor_numerico"]
].copy()

base["Valor_numerico"] = base["Valor_numerico"] / 1000

base.columns = [
    "Cenário",
    "Ano",
    "Conta",
    "Valor (R$ mil)"
]


indicadores = kpis_final.copy()

# Entrega 2: confere se os novos indicadores chegaram até aqui
faltando = [c for c in colunas_novos_indicadores if c not in indicadores.columns]

if faltando:
    raise KeyError(
        f"Indicadores ausentes no kpis_final: {faltando}. "
        "Execute as células 'Consolidação dos KPIs' e 'Consolidação dos Indicadores', nessa ordem."
    )

# Valores em reais passam para R$ mil (a Entrega 2 inclui CCL, FCO e FCL)
for col in [
    "NOPAT",
    "Capital Investido",
    "EVA",
    "MVA Aproximado",
    "CCL",
    "Necessidade de Capital de Giro",
    "Recebimentos Estimados",
    "FCO",
    "FCL"
]:
    indicadores[col] = indicadores[col] / 1000

# Divisões por zero não podem ir para o Excel
indicadores = indicadores.replace([np.inf, -np.inf], np.nan)


ebitda_base = (
    df_kpis[
        df_kpis["Conta"] == "DRE - EBITDA"
    ][["Cenario", "Ano", "Valor_numerico"]]
    .copy()
)

ebitda_base["EBITDA"] = (
    ebitda_base["Valor_numerico"] / 1000
)

ebitda_base = ebitda_base[
    ["Cenario", "Ano", "EBITDA"]
]

indicadores = indicadores.merge(
    ebitda_base,
    on=["Cenario", "Ano"],
    how="left"
)


cenarios = sorted(
    indicadores["Cenario"].unique()
)

anos = [
    f"Ano {i}"
    for i in range(1, 13)
]


dicionario = pd.DataFrame([
    [
        "Margem EBITDA",
        "EBITDA / Receita Líquida",
        "%",
        "Indica quanto da receita líquida é convertida em EBITDA."
    ],
    [
        "Margem Líquida",
        "Resultado Líquido / Receita Líquida",
        "%",
        "Indica quanto da receita líquida permanece como resultado líquido."
    ],
    [
        "Liquidez Corrente (LC)",
        "Ativo Circulante / Passivo Circulante",
        "Índice",
        "Avalia a capacidade de pagamento das obrigações de curto prazo."
    ],
    [
        "Liquidez Seca (LS)",
        "(Ativo Circulante - Estoques) / Passivo Circulante",
        "Índice",
        "Avalia a liquidez de curto prazo sem considerar os estoques."
    ],
    [
        "Liquidez Imediata (LI)",
        "Disponível / Passivo Circulante",
        "Índice",
        "Avalia a capacidade de pagamento imediato das obrigações de curto prazo."
    ],
    [
        "Liquidez Geral (LG)",
        "(AC + RLP) / (PC + ELP)",
        "Índice",
        "Avalia a capacidade de pagamento considerando curto e longo prazo."
    ],
    [
        "Participação de Capital de Terceiros (PCT)",
        "(PC + ELP) / Patrimônio Líquido",
        "Índice",
        "Mostra a relação entre recursos de terceiros e patrimônio líquido."
    ],
    [
        "Composição do Endividamento (CE)",
        "PC / (PC + ELP)",
        "%",
        "Mostra a parcela do endividamento concentrada no curto prazo."
    ],
    [
        "Imobilização do Patrimônio Líquido (IPL)",
        "Permanente / Patrimônio Líquido",
        "Índice",
        "Indica quanto do patrimônio líquido está aplicado no permanente."
    ],
    [
        "EBITDA",
        "Valor disponibilizado na base",
        "R$ mil",
        "Resultado antes de juros, impostos, depreciação e amortização."
    ],
    [
        "NOPAT",
        "Resultado Operacional - Imposto estimado sobre o Resultado Operacional",
        "R$ mil",
        "Resultado operacional após a estimativa dos impostos."
    ],
    [
        "Capital Investido",
        "Patrimônio Líquido + Dívidas Financeiras - Disponível",
        "R$ mil",
        "Capital utilizado como base para o cálculo da criação de valor."
    ],
    [
        "EVA",
        "NOPAT - (Capital Investido × WACC)",
        "R$ mil",
        "Indica o valor econômico criado após remunerar o capital investido."
    ],
    [
        "MVA Aproximado",
        "EVA / WACC",
        "R$ mil",
        "Estimativa do valor agregado baseada na perpetuidade do EVA."
    ],
    [
        "ROIC",
        "NOPAT / Capital Investido",
        "%",
        "Indicador auxiliar utilizado na análise do retorno sobre o capital investido."
    ],
    [
        "WACC",
        "Custo médio ponderado de capital",
        "%",
        "Parâmetro auxiliar de 8% utilizado nos cálculos de EVA e MVA."
    ]
], columns=[
    "Indicador",
    "Fórmula / critério",
    "Unidade",
    "Interpretação"
])


novos_indicadores = pd.DataFrame([
    [
        "ROE",
        "Resultado Líquido / Patrimônio Líquido Médio",
        "%",
        "Mede o retorno obtido sobre o capital próprio dos acionistas."
    ],
    [
        "DuPont - Margem Líquida",
        "Resultado Líquido / Receita Líquida",
        "%",
        "Mostra quanto da receita líquida permanece como resultado."
    ],
    [
        "DuPont - Giro do Ativo",
        "Receita Líquida / Ativo Total Médio",
        "Índice",
        "Mede quanto de receita é gerado por unidade de ativo."
    ],
    [
        "DuPont - Alavancagem",
        "Ativo Total Médio / Patrimônio Líquido Médio",
        "Índice",
        "Mostra a relação entre os ativos e o capital próprio."
    ],
    [
        "ROE DuPont",
        "Margem Líquida × Giro do Ativo × Alavancagem",
        "%",
        "Decompõe o retorno sobre o patrimônio líquido em três fatores."
    ],
    [
        "CCL",
        "Ativo Circulante − Passivo Circulante",
        "R$ mil",
        "Indica a folga financeira de curto prazo, usando o passivo informado."
    ],
    [
        "PMR",
        "Contas a Receber de Clientes / Receita Operacional × 360",
        "Dias",
        "Estima o prazo médio de recebimento das vendas."
    ],
    [
        "PMP",
        "Fornecedores / Custos × 360",
        "Dias",
        "Estima o prazo médio de pagamento aos fornecedores."
    ],
    [
        "Ciclo Financeiro",
        "PMR − PMP",
        "Dias",
        "Indica o intervalo entre pagar fornecedores e receber dos clientes."
    ],
    [
        "Necessidade de Capital de Giro",
        "Ciclo Financeiro × Custos / 360",
        "R$ mil",
        "Estimativa didática do capital necessário para financiar o ciclo."
    ],
    [
        "Taxa de Conversão da Receita em Caixa",
        "Recebimentos Estimados / Receita",
        "%",
        "Estima quanto da receita do período se converteu em recebimentos."
    ],
    [
        "FCO",
        "Recebimentos − Tributos pagos − Custos pagos − IR/CS pago",
        "R$ mil",
        "Estima o fluxo de caixa operacional com as contas disponíveis."
    ],
    [
        "FCO / EBITDA",
        "FCO / EBITDA",
        "%",
        "Mede quanto do EBITDA se converte em caixa operacional."
    ],
    [
        "FCO / Lucro Líquido",
        "FCO / Lucro Líquido",
        "Índice",
        "Avalia a conversão do lucro contábil em caixa."
    ],
    [
        "FCL",
        "FCO − Investimentos",
        "R$ mil",
        "Estima o caixa livre após os investimentos."
    ],
    [
        "Capex / Receita",
        "Investimentos / Receita",
        "%",
        "Mede a parcela da receita destinada a investimentos."
    ],
    [
        "Capex / D&A",
        "Investimentos / Depreciação e Amortização",
        "Índice",
        "Compara o investimento com o desgaste dos ativos."
    ],
    [
        "Cobertura de Juros",
        "FCO / Juros pagos",
        "Índice",
        "Mede a capacidade do caixa operacional de cobrir os juros."
    ],
    [
        "FCO / Dívida Bruta",
        "FCO / (Empréstimos CP + LP)",
        "%",
        "Compara o caixa operacional com o saldo de dívidas financeiras."
    ],
    [
        "Payout em Caixa",
        "Distribuição a Acionistas / Lucro Líquido",
        "%",
        "Compara a distribuição informada aos acionistas com o lucro."
    ],
    [
        "Cobertura de Dividendos",
        "FCL / Distribuição a Acionistas",
        "Índice",
        "Avalia se o caixa livre cobre as distribuições informadas."
    ]
], columns=dicionario.columns)

dicionario = pd.concat(
    [dicionario, novos_indicadores],
    ignore_index=True
)


print("Gravando dados...")


with pd.ExcelWriter(
    arquivo_saida,
    engine="xlsxwriter"
) as writer:

    dicionario.to_excel(
        writer,
        sheet_name="Dicionário de KPIs financeiros",
        index=False,
        startrow=3
    )

    base.to_excel(
        writer,
        sheet_name="Dados de base",
        index=False,
        startrow=3
    )

    indicadores.to_excel(
        writer,
        sheet_name="_Indicadores",
        index=False
    )


print("Aplicando layout...")


wb = load_workbook(arquivo_saida)

ws_dic = wb["Dicionário de KPIs financeiros"]
ws_base = wb["Dados de base"]
ws_aux = wb["_Indicadores"]

ws_demo = wb.create_sheet("Demonstrações contábeis")
ws_ind = wb.create_sheet("Indicadores - Índices")
ws_listas = wb.create_sheet("_Listas")


GRAFITE = "403D46"
TEXTO = "504C57"

LAVANDA_ESCURO = "756C86"
LAVANDA = "AAA1BA"
LAVANDA_MEDIA = "D7D1E0"
LAVANDA_CLARA = "F1EEF5"

CINZA = "F8F7F9"
BRANCO = "FFFFFF"
BORDA = "DDD8E3"

VERDE = "E2F0E5"
VERDE_TEXTO = "263D2D"

VERMELHO = "F6E3E3"
VERMELHO_TEXTO = "573436"


fill_lavanda_escuro = PatternFill(
    "solid",
    fgColor=LAVANDA_ESCURO
)

fill_lavanda = PatternFill(
    "solid",
    fgColor=LAVANDA
)

fill_lavanda_media = PatternFill(
    "solid",
    fgColor=LAVANDA_MEDIA
)

fill_lavanda_clara = PatternFill(
    "solid",
    fgColor=LAVANDA_CLARA
)

fill_cinza = PatternFill(
    "solid",
    fgColor=CINZA
)

fill_verde = PatternFill(
    "solid",
    fgColor=VERDE
)

fill_vermelho = PatternFill(
    "solid",
    fgColor=VERMELHO
)

borda_fina = Side(
    style="thin",
    color=BORDA
)


def titulo(ws, texto, subtitulo):

    ws.sheet_view.showGridLines = False

    ws.merge_cells("A1:H1")
    ws["A1"] = texto

    ws["A1"].font = Font(
        name="Aptos Display",
        size=20,
        bold=True,
        color=GRAFITE
    )

    ws["A1"].alignment = Alignment(
        vertical="center"
    )

    ws.row_dimensions[1].height = 32

    ws.merge_cells("A2:H2")
    ws["A2"] = subtitulo

    ws["A2"].font = Font(
        name="Aptos",
        size=10,
        color=TEXTO
    )


def cabecalho(ws, linha, inicio, fim):

    for col in range(inicio, fim + 1):

        c = ws.cell(linha, col)

        c.fill = fill_lavanda_escuro

        c.font = Font(
            name="Aptos",
            size=10,
            bold=True,
            color=BRANCO
        )

        c.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True
        )

        c.border = Border(
            bottom=borda_fina
        )


def seletor(ws):

    ws["A4"] = "CENÁRIO"

    ws["A4"].font = Font(
        name="Aptos",
        size=9,
        bold=True,
        color=LAVANDA_ESCURO
    )

    ws["B4"] = cenarios[0]

    ws["B4"].fill = fill_lavanda_clara

    ws["B4"].font = Font(
        name="Aptos",
        size=10,
        bold=True,
        color=GRAFITE
    )

    ws["B4"].alignment = Alignment(
        horizontal="center"
    )

    ws["B4"].border = Border(
        left=borda_fina,
        right=borda_fina,
        top=borda_fina,
        bottom=borda_fina
    )

    dv = DataValidation(
        type="list",
        formula1="=ListaCenarios",
        allow_blank=False
    )

    ws.add_data_validation(dv)
    dv.add(ws["B4"])


for i, cenario in enumerate(
    cenarios,
    1
):
    ws_listas.cell(
        i,
        1,
        cenario
    )


wb.defined_names.add(
    DefinedName(
        "ListaCenarios",
        attr_text=f"'_Listas'!$A$1:$A${len(cenarios)}"
    )
)


titulo(
    ws_dic,
    "Dicionário de KPIs financeiros",
    "Definições, fórmulas e interpretação dos indicadores utilizados."
)

cabecalho(
    ws_dic,
    4,
    1,
    4
)


ws_dic.column_dimensions["A"].width = 40
ws_dic.column_dimensions["B"].width = 58
ws_dic.column_dimensions["C"].width = 15
ws_dic.column_dimensions["D"].width = 70


for linha in range(
    5,
    ws_dic.max_row + 1
):

    ws_dic.row_dimensions[linha].height = 40

    for col in range(1, 5):

        c = ws_dic.cell(
            linha,
            col
        )

        c.alignment = Alignment(
            vertical="center",
            wrap_text=True
        )

        c.border = Border(
            bottom=borda_fina
        )

        if linha % 2 == 0:
            c.fill = fill_lavanda_clara


ws_dic.freeze_panes = "A5"


titulo(
    ws_demo,
    "Demonstrações contábeis",
    "Selecione um cenário para visualizar BP, DRE e fluxo de caixa."
)

seletor(ws_demo)


ordem_contas = [
    "BAL - Ativo Circulante",
    "BAL - Disponível",
    "BAL - Contas a Receber - Clientes",
    "BAL - Contas a Receber - Partes Relacionadas",
    "BAL - Contas a Receber - SWAP",
    "BAL - Estoques Diversos",
    "BAL - Créditos Tributários",
    "BAL - Outros Créditos",
    "BAL - Realizável a Longo Prazo",
    "BAL - Outros Créditos LP",
    "BAL - At Fiscal Diferido",
    "BAL - Investimentos - Imobilizado",
    "BAL - Investimentos - Intangível",
    "BAL - Outorga da Concessão",
    "BAL - Diferido",
    "BAL - Amortização - Intangível",
    "BAL - Amortização Acumulada",
    "BAL - Depreciação Acumulada",
    "BAL - Permanente",
    "BAL - Total do Ativo",
    "BAL - Passivo Circulante",
    "BAL - Fornecedores",
    "BAL - Contas a Pagar - Parte Relacionada",
    "BAL - Emprést",
    "BAL - Empréstimos",
    "BAL - Tributos a pagar",
    "BAL - Impostos",
    "BAL - Encargos Sociais e Trabalhistas",
    "BAL - Obrigações com o Poder Concedente",
    "BAL - Provisão Manutenção",
    "BAL - Prov para Contingências",
    "BAL - Outros Débitos",
    "BAL - Outros deb",
    "BAL - Exigível a Longo Prazo",
    "BAL - Total do Passivo",
    "BAL - Patrimônio Líquido",
    "BAL - Capital Social",
    "BAL - Reservas Legais",
    "BAL - Reservas de Capital",
    "BAL - Reserva de Retenção de Lucros",
    "BAL - Resultado Acumulado",
    "BAL - Resultado do Período",
    "BAL - Dividendos Antecipados",

    "DRE - Receita",
    "DRE - Tributos",
    "DRE - Custos",
    "DRE - Depreciação e Amortização",
    "DRE - Resultado Operacional",
    "DRE - Outros Resultados Operacionais",
    "DRE - Resultado Financeiro",
    "DRE - Resultado Antes do Imposto de Renda",
    "DRE - Imposto de Renda e Contribuição Social",
    "DRE - Resultado Líquido",
    "DRE - Resultado Líquido após Equivalência",
    "DRE - Receitas Financeiras",
    "DRE - Despesas Financeiras",
    "DRE - EBITDA",

    "FLU - Entradas",
    "FLU - Receita",
    "FLU - Tributos",
    "FLU - Custos",
    "FLU - Resultado Financeiro",
    "FLU - Despesas Financeiras",
    "FLU - Imposto de Renda e Contribuição Social",
    "FLU - Geração de Caixa",
    "FLU - Investimentos",
    "FLU - Distribuição para Acionista",
    "FLU - Saldo Inicial",
    "FLU - Saldo Final"
]


ws_demo["A6"] = "Conta"

for col, ano in enumerate(
    anos,
    2
):

    ws_demo.cell(
        6,
        col,
        f"{ano}\n(R$ mil)"
    )


cabecalho(
    ws_demo,
    6,
    1,
    13
)

ws_demo.row_dimensions[6].height = 40


linha = 7

secoes = [
    ("BALANÇO PATRIMONIAL", "BAL -"),
    ("DEMONSTRAÇÃO DO RESULTADO", "DRE -"),
    ("FLUXO DE CAIXA", "FLU -")
]


for nome_secao, prefixo in secoes:

    ws_demo.merge_cells(
        start_row=linha,
        start_column=1,
        end_row=linha,
        end_column=13
    )

    c = ws_demo.cell(
        linha,
        1
    )

    c.value = nome_secao
    c.fill = fill_lavanda

    c.font = Font(
        name="Aptos",
        size=10,
        bold=True,
        color=GRAFITE
    )

    ws_demo.row_dimensions[linha].height = 24

    linha += 1


    contas_secao = [
        conta
        for conta in ordem_contas
        if conta.startswith(prefixo)
    ]


    for conta in contas_secao:

        ws_demo.cell(
            linha,
            1,
            conta.replace(prefixo, "")
        )


        for col, ano in enumerate(
            anos,
            2
        ):

            formula = (
                f'=IF('
                f'COUNTIFS('
                f"'Dados de base'!$A:$A,$B$4,"
                f"'Dados de base'!$B:$B,\"{ano}\","
                f"'Dados de base'!$C:$C,\"{conta}\""
                f')=0,'
                f'"N/D",'
                f'SUMIFS('
                f"'Dados de base'!$D:$D,"
                f"'Dados de base'!$A:$A,$B$4,"
                f"'Dados de base'!$B:$B,\"{ano}\","
                f"'Dados de base'!$C:$C,\"{conta}\""
                f'))'
            )

            ws_demo.cell(
                linha,
                col
            ).value = formula

            ws_demo.cell(
                linha,
                col
            ).number_format = '#,##0.00'


        if any(
            x in conta
            for x in [
                "Total do Ativo",
                "Total do Passivo",
                "Patrimônio Líquido",
                "Resultado Líquido",
                "EBITDA",
                "Geração de Caixa",
                "Saldo Final"
            ]
        ):

            for col in range(
                1,
                14
            ):

                ws_demo.cell(
                    linha,
                    col
                ).fill = fill_lavanda_clara

                ws_demo.cell(
                    linha,
                    col
                ).font = Font(
                    name="Aptos",
                    size=10,
                    bold=True,
                    color=GRAFITE
                )


        linha += 1


ws_demo.column_dimensions["A"].width = 43

for col in range(
    2,
    14
):

    ws_demo.column_dimensions[
        get_column_letter(col)
    ].width = 16


ws_demo.freeze_panes = "B7"


titulo(
    ws_ind,
    "Indicadores / Índices financeiros",
    "Selecione um cenário para acompanhar os indicadores ao longo dos 12 anos."
)

seletor(ws_ind)


ws_ind["A6"] = "Indicador"

for col, ano in enumerate(
    anos,
    2
):

    ws_ind.cell(
        6,
        col,
        ano
    )


cabecalho(
    ws_ind,
    6,
    1,
    13
)

ws_ind.row_dimensions[6].height = 34


estrutura_indicadores = [
    ("Margem EBITDA (%)", "Margem EBITDA", "percentual"),
    ("Margem Líquida (%)", "Margem Líquida", "percentual"),

    ("Liquidez Corrente (LC)", "LC", "indice"),
    ("Liquidez Seca (LS)", "LS", "indice"),
    ("Liquidez Imediata (LI)", "LI", "indice"),
    ("Liquidez Geral (LG)", "LG", "indice"),

    (
        "Participação de Capital de Terceiros (PCT)",
        "PCT",
        "indice"
    ),

    (
        "Composição do Endividamento (CE)",
        "CE",
        "percentual"
    ),

    (
        "Imobilização do Patrimônio Líquido (IPL)",
        "IPL",
        "indice"
    ),

    ("EBITDA (R$ mil)", "EBITDA", "moeda"),
    ("NOPAT (R$ mil)", "NOPAT", "moeda"),

    (
        "Capital Investido (R$ mil)",
        "Capital Investido",
        "moeda"
    ),

    ("EVA (R$ mil)", "EVA", "moeda"),

    (
        "MVA Aproximado (R$ mil)",
        "MVA Aproximado",
        "moeda"
    ),

    # Novos indicadores - Entrega 2
    ("ROE (%)", "ROE", "percentual"),
    ("DuPont - Margem Líquida (%)", "DuPont - Margem Líquida", "percentual"),
    ("DuPont - Giro do Ativo", "DuPont - Giro do Ativo", "indice"),
    ("DuPont - Alavancagem", "DuPont - Alavancagem", "indice"),
    ("ROE DuPont (%)", "ROE DuPont", "percentual"),

    ("CCL (R$ mil)", "CCL", "moeda"),
    ("PMR (dias)", "PMR", "indice"),
    ("PMP (dias)", "PMP", "indice"),
    ("Ciclo Financeiro (dias)", "Ciclo Financeiro", "indice"),
    (
        "Necessidade de Capital de Giro (R$ mil)",
        "Necessidade de Capital de Giro",
        "moeda"
    ),

    (
        "Taxa de Conversão da Receita em Caixa (%)",
        "Taxa de Conversão da Receita em Caixa",
        "percentual"
    ),
    ("FCO (R$ mil)", "FCO", "moeda"),
    ("FCO / EBITDA (%)", "FCO / EBITDA", "percentual"),
    ("FCO / Lucro Líquido", "FCO / Lucro Líquido", "indice"),
    ("FCL (R$ mil)", "FCL", "moeda"),

    ("Capex / Receita (%)", "Capex / Receita", "percentual"),
    ("Capex / D&A", "Capex / D&A", "indice"),
    ("Cobertura de Juros", "Cobertura de Juros", "indice"),
    ("FCO / Dívida Bruta (%)", "FCO / Dívida Bruta", "percentual"),
    ("Payout em Caixa (%)", "Payout em Caixa", "percentual"),
    ("Cobertura de Dividendos", "Cobertura de Dividendos", "indice")
]


headers_ind = list(
    indicadores.columns
)


linhas_indicadores = {}


for linha, (
    nome_exibicao,
    nome_coluna,
    tipo
) in enumerate(
    estrutura_indicadores,
    7
):

    linhas_indicadores[
        nome_coluna
    ] = linha

    ws_ind.cell(
        linha,
        1,
        nome_exibicao
    )

    ws_ind.cell(
        linha,
        1
    ).font = Font(
        name="Aptos",
        size=10,
        bold=True,
        color=GRAFITE
    )


    if linha % 2 == 0:

        for col in range(
            1,
            14
        ):

            ws_ind.cell(
                linha,
                col
            ).fill = fill_lavanda_clara


    letra = get_column_letter(
        headers_ind.index(
            nome_coluna
        ) + 1
    )


    for col, ano in enumerate(
        anos,
        2
    ):

        celula = ws_ind.cell(
            linha,
            col
        )

        col_ano = get_column_letter(col)

        formula_contagem = (
            f'COUNTIFS('
            f"'_Indicadores'!$A:$A,$B$4,"
            f"'_Indicadores'!$B:$B,{col_ano}$6,"
            f"'_Indicadores'!${letra}:${letra},\"<>\""
            f')'
        )

        formula_valor = (
            f'SUMIFS('
            f"'_Indicadores'!${letra}:${letra},"
            f"'_Indicadores'!$A:$A,$B$4,"
            f"'_Indicadores'!$B:$B,{col_ano}$6"
            f')'
        )

        celula.value = f'=IF({formula_contagem}=0,"N/D",{formula_valor})'


        if tipo == "percentual":
            celula.number_format = "0.00%"

        elif tipo == "indice":
            celula.number_format = "0.00"

        elif tipo == "moeda":
            celula.number_format = '#,##0.00'


        celula.alignment = Alignment(
            horizontal="center",
            vertical="center"
        )


def aplicar_fundo(
    linha,
    condicao
):

    for col in range(
        2,
        14
    ):

        valor = None

        nome_coluna = None

        for nome, lin in linhas_indicadores.items():
            if lin == linha:
                nome_coluna = nome
                break

        if nome_coluna is None:
            continue

        ano = anos[
            col - 2
        ]

        filtro = indicadores[
            (indicadores["Cenario"] == cenarios[0]) &
            (indicadores["Ano"] == ano)
        ]

        if filtro.empty:
            continue

        valor = filtro[
            nome_coluna
        ].iloc[0]

        if pd.isna(valor):
            continue

        resultado = condicao(
            valor
        )

        if resultado == "ruim":

            ws_ind.cell(
                linha,
                col
            ).fill = fill_vermelho

            ws_ind.cell(
                linha,
                col
            ).font = Font(
                name="Aptos",
                size=10,
                color=VERMELHO_TEXTO
            )

        elif resultado == "bom":

            ws_ind.cell(
                linha,
                col
            ).fill = fill_verde

            ws_ind.cell(
                linha,
                col
            ).font = Font(
                name="Aptos",
                size=10,
                color=VERDE_TEXTO
            )


for nome in [
    "LC",
    "LS",
    "LG"
]:

    aplicar_fundo(
        linhas_indicadores[nome],
        lambda x:
        "ruim" if x < 1
        else "bom" if x >= 1.5
        else None
    )


aplicar_fundo(
    linhas_indicadores["LI"],
    lambda x:
    "ruim" if x < 0.5
    else "bom" if x >= 1
    else None
)


aplicar_fundo(
    linhas_indicadores["PCT"],
    lambda x:
    "ruim" if x >= 4
    else "bom" if x <= 1
    else None
)


aplicar_fundo(
    linhas_indicadores["CE"],
    lambda x:
    "ruim" if x >= 0.6
    else "bom" if x <= 0.3
    else None
)


aplicar_fundo(
    linhas_indicadores["IPL"],
    lambda x:
    "ruim" if x >= 1
    else "bom" if x <= 0.5
    else None
)


aplicar_fundo(
    linhas_indicadores["EVA"],
    lambda x:
    "ruim" if x < 0
    else "bom" if x > 0
    else None
)


aplicar_fundo(
    linhas_indicadores["MVA Aproximado"],
    lambda x:
    "ruim" if x < 0
    else "bom" if x > 0
    else None
)


# Entrega 2: cores dos novos indicadores
# Aqui é usada formatação condicional do Excel, que acompanha o cenário
# escolhido no seletor (as cores são recalculadas quando o cenário muda).

from openpyxl.formatting.rule import FormulaRule

dxf_vermelho = PatternFill(
    start_color=VERMELHO,
    end_color=VERMELHO,
    fill_type="solid"
)

dxf_verde = PatternFill(
    start_color=VERDE,
    end_color=VERDE,
    fill_type="solid"
)


def cor_condicional(nome_coluna, limite_ruim, limite_bom):

    linha = linhas_indicadores[nome_coluna]
    ref = f"B{linha}"
    faixa = f"B{linha}:M{linha}"

    # ISNUMBER evita pintar as células "N/D"
    ws_ind.conditional_formatting.add(
        faixa,
        FormulaRule(
            formula=[f"AND(ISNUMBER({ref}),{ref}<{limite_ruim})"],
            fill=dxf_vermelho,
            font=Font(color=VERMELHO_TEXTO)
        )
    )

    ws_ind.conditional_formatting.add(
        faixa,
        FormulaRule(
            formula=[f"AND(ISNUMBER({ref}),{ref}>={limite_bom})"],
            fill=dxf_verde,
            font=Font(color=VERDE_TEXTO)
        )
    )


# Resultado / caixa: negativo = atenção, positivo = favorável
for nome in ["ROE", "ROE DuPont", "CCL", "FCO", "FCL"]:
    cor_condicional(nome, 0, 0.0000001)

# Coberturas: abaixo de 1 o caixa não cobre a obrigação
for nome in ["Cobertura de Juros", "Cobertura de Dividendos"]:
    cor_condicional(nome, 1, 1)


ws_ind.column_dimensions["A"].width = 45

for col in range(
    2,
    14
):

    ws_ind.column_dimensions[
        get_column_letter(col)
    ].width = 16


for linha in range(
    7,
    7 + len(estrutura_indicadores)
):

    ws_ind.row_dimensions[
        linha
    ].height = 25


ws_ind.freeze_panes = "B7"


titulo(
    ws_base,
    "Dados de base",
    "Base tratada utilizada para os cálculos e análises financeiras."
)

cabecalho(
    ws_base,
    4,
    1,
    4
)


ws_base.freeze_panes = "A5"

ws_base.column_dimensions["A"].width = 22
ws_base.column_dimensions["B"].width = 12
ws_base.column_dimensions["C"].width = 48
ws_base.column_dimensions["D"].width = 20


for cell in ws_base["D"][4:]:
    cell.number_format = '#,##0.00'


for ws in [
    ws_dic,
    ws_demo,
    ws_ind,
    ws_base
]:

    ws.sheet_view.showGridLines = False

    ws.sheet_properties.pageSetUpPr.fitToPage = True

    ws.page_setup.orientation = "landscape"

    ws.page_setup.fitToWidth = 1

    ws.page_setup.fitToHeight = 0


ws_aux.sheet_state = "hidden"
ws_listas.sheet_state = "hidden"


wb._sheets = [
    ws_dic,
    ws_demo,
    ws_ind,
    ws_base,
    ws_aux,
    ws_listas
]


wb.calculation.fullCalcOnLoad = True
wb.calculation.forceFullCalc = True
wb.calculation.calcMode = "auto"


wb.save(
    arquivo_saida
)


print("Planilha criada com sucesso!")
print(arquivo_saida.resolve())

## Ranking dos Cenários e Anos

Nesta etapa, os indicadores financeiros calculados anteriormente são utilizados para identificar as combinações de cenário e ano que apresentaram os melhores resultados.

Como os indicadores possuem unidades e escalas diferentes, seus valores não são comparados diretamente. Para tornar a avaliação possível, cada indicador é convertido em uma pontuação relativa de 0 a 100, considerando sua posição em relação aos demais resultados da base.

Para indicadores em que valores maiores representam um resultado mais favorável, a pontuação aumenta conforme o desempenho. Para os indicadores relacionados à estrutura de capital utilizados nesta análise, a pontuação é invertida, atribuindo maior pontuação aos menores valores.

A pontuação final é obtida pela média das pontuações dos indicadores considerados, utilizando o mesmo peso para todos eles. Ao final, as combinações são ordenadas pela pontuação obtida para identificar o Top 10 de cenários e anos.

In [7]:
ranking = kpis_final.copy()

# Indicadores em que valores maiores recebem maior pontuação
maior_melhor = [
    "Margem EBITDA",
    "Margem Líquida",
    "LC",
    "LS",
    "LI",
    "LG",
    "EVA",
    "MVA Aproximado"
]

# Indicadores em que valores menores recebem maior pontuação
menor_melhor = [
    "PCT",
    "CE",
    "IPL"
]

# Pontuação de 0 a 100 para cada indicador
for indicador in maior_melhor:
    ranking[f"Score {indicador}"] = (
        ranking[indicador]
        .rank(pct=True, method="average") * 100
    )

for indicador in menor_melhor:
    ranking[f"Score {indicador}"] = (
        (1 - ranking[indicador].rank(pct=True, method="average")) * 100
    )

colunas_score = [
    f"Score {indicador}"
    for indicador in maior_melhor + menor_melhor
]

# Exige que todos os indicadores usados no ranking estejam disponíveis
ranking["Indicadores válidos"] = ranking[colunas_score].notna().sum(axis=1)

ranking_validos = ranking[
    ranking["Indicadores válidos"] == len(colunas_score)
].copy()

# Todos os indicadores recebem o mesmo peso nesta primeira metodologia
ranking_validos["Pontuação Final"] = (
    ranking_validos[colunas_score].mean(axis=1)
)

ranking_validos = ranking_validos.sort_values(
    "Pontuação Final",
    ascending=False
).reset_index(drop=True)

ranking_validos["Posição"] = ranking_validos.index + 1

top10 = ranking_validos[
    [
        "Posição",
        "Cenario",
        "Ano",
        "Pontuação Final",
        "Margem EBITDA",
        "Margem Líquida",
        "LC",
        "LS",
        "LI",
        "LG",
        "PCT",
        "CE",
        "IPL",
        "EVA",
        "MVA Aproximado"
    ]
].head(10)

display(top10)

,Posição,Cenario,Ano,Pontuação Final,Margem EBITDA,Margem Líquida,LC,LS,LI,LG,PCT,CE,IPL,EVA,MVA Aproximado
0,1,Total Cen_01128,Ano 11,86.102388,0.914008,0.530541,1.865615,1.865596,1.872130,0.903501,1.346467,0.371831,0.128618,4.910889e+09,6.138611e+10
1,2,Total Cen_01127,Ano 11,86.020489,0.913483,0.529370,1.781133,1.781114,1.787648,0.872110,1.405889,0.371831,0.134294,4.832783e+09,6.040978e+10
2,3,Total Cen_01126,Ano 11,85.874082,0.912952,0.528184,1.697311,1.697292,1.703826,0.840965,1.470267,0.371831,0.140443,4.755497e+09,5.944371e+10
3,4,Total Cen_01128,Ano 9,85.254075,0.907076,0.480585,1.827726,1.827695,1.725630,0.733307,1.734699,0.273995,0.512034,3.736570e+09,4.670713e+10
4,5,Total Cen_01127,Ano 9,84.869605,0.906578,0.479142,1.746243,1.746212,1.644148,0.711007,1.804504,0.273995,0.532639,3.680666e+09,4.600833e+10
5,6,Total Cen_01128,Ano 10,84.522727,0.912129,0.504078,1.627058,1.627036,1.590875,0.764697,1.566764,0.343711,0.291636,4.264055e+09,5.330068e+10
6,7,Total Cen_01125,Ano 11,84.386880,0.912414,0.526984,1.614145,1.614126,1.620660,0.810063,1.540246,0.371831,0.147128,4.679023e+09,5.848779e+10
7,8,Total Cen_01127,Ano 10,84.309860,0.911631,0.502733,1.551463,1.551441,1.515280,0.738737,1.633190,0.343711,0.304001,4.197702e+09,5.247128e+10
8,9,Total Cen_01124,Ano 11,84.161961,0.911869,0.525770,1.531629,1.531610,1.538144,0.779404,1.616586,0.371831,0.154420,4.603354e+09,5.754192e+10
9,10,Total Cen_01126,Ano 10,84.063820,0.911127,0.501373,1.476406,1.476384,1.440223,0.712962,1.704960,0.343711,0.317360,4.131984e+09,5.164980e+10


### Pontuação por Dimensão

Para facilitar a interpretação do ranking, os indicadores foram agrupados em quatro dimensões: rentabilidade, liquidez, estrutura de capital e geração de valor.

A pontuação de cada dimensão corresponde à média das pontuações dos indicadores que fazem parte daquele grupo. Dessa forma, é possível identificar quais aspectos financeiros contribuíram mais para a posição de cada combinação de cenário e ano no ranking.

In [8]:
ranking_validos["Rentabilidade"] = ranking_validos[
    [
        "Score Margem EBITDA",
        "Score Margem Líquida"
    ]
].mean(axis=1)

ranking_validos["Liquidez"] = ranking_validos[
    [
        "Score LC",
        "Score LS",
        "Score LI",
        "Score LG"
    ]
].mean(axis=1)

ranking_validos["Estrutura de Capital"] = ranking_validos[
    [
        "Score PCT",
        "Score CE",
        "Score IPL"
    ]
].mean(axis=1)

ranking_validos["Geração de Valor"] = ranking_validos[
    [
        "Score EVA",
        "Score MVA Aproximado"
    ]
].mean(axis=1)

top10_resumo = ranking_validos[
    [
        "Posição",
        "Cenario",
        "Ano",
        "Rentabilidade",
        "Liquidez",
        "Estrutura de Capital",
        "Geração de Valor",
        "Pontuação Final"
    ]
].head(10).copy()

top10_resumo[
    [
        "Rentabilidade",
        "Liquidez",
        "Estrutura de Capital",
        "Geração de Valor",
        "Pontuação Final"
    ]
] = top10_resumo[
    [
        "Rentabilidade",
        "Liquidez",
        "Estrutura de Capital",
        "Geração de Valor",
        "Pontuação Final"
    ]
].round(2)

display(top10_resumo)

,Posição,Cenario,Ano,Rentabilidade,Liquidez,Estrutura de Capital,Geração de Valor,Pontuação Final
0,1,Total Cen_01128,Ano 11,91.56,90.26,67.66,100.00,86.10
1,2,Total Cen_01127,Ano 11,91.16,90.24,67.65,99.99,86.02
2,3,Total Cen_01126,Ano 11,90.50,90.17,67.65,99.98,85.87
3,4,Total Cen_01128,Ano 9,83.76,90.23,70.67,98.68,85.25
4,5,Total Cen_01127,Ano 9,83.69,90.18,70.62,96.80,84.87
5,6,Total Cen_01128,Ano 10,87.57,86.48,69.61,99.93,84.52
6,7,Total Cen_01125,Ano 11,89.72,86.48,67.64,99.98,84.39
7,8,Total Cen_01127,Ano 10,87.55,86.38,68.99,99.91,84.31
8,9,Total Cen_01124,Ano 11,88.75,86.36,67.64,99.97,84.16
9,10,Total Cen_01126,Ano 10,87.53,86.29,68.23,99.89,84.06


### Preparação dos Dados para o Ranking

Após o cálculo das pontuações, os resultados do ranking são organizados para apresentação na planilha final.

A tabela reúne a posição de cada combinação de cenário e ano, as pontuações obtidas nas quatro dimensões analisadas e os principais indicadores financeiros utilizados no cálculo. Essa organização permite visualizar o resultado geral do ranking e, ao mesmo tempo, consultar os valores que contribuíram para cada posição.

In [9]:
ranking_excel = ranking_validos[
    [
        "Posição",
        "Cenario",
        "Ano",
        "Rentabilidade",
        "Liquidez",
        "Estrutura de Capital",
        "Geração de Valor",
        "Pontuação Final",
        "Margem EBITDA",
        "Margem Líquida",
        "LC",
        "LS",
        "LI",
        "LG",
        "PCT",
        "CE",
        "IPL",
        "EVA",
        "MVA Aproximado"
    ]
].copy()

ranking_excel = ranking_excel.rename(
    columns={"Cenario": "Cenário"}
)

ranking_excel.head(10)

,Posição,Cenário,Ano,Rentabilidade,Liquidez,Estrutura de Capital,Geração de Valor,Pontuação Final,Margem EBITDA,Margem Líquida,LC,LS,LI,LG,PCT,CE,IPL,EVA,MVA Aproximado
0,1,Total Cen_01128,Ano 11,91.559028,90.257734,67.659091,100.000000,86.102388,0.914008,0.530541,1.865615,1.865596,1.872130,0.903501,1.346467,0.371831,0.128618,4.910889e+09,6.138611e+10
1,2,Total Cen_01127,Ano 11,91.159722,90.239741,67.654040,99.992424,86.020489,0.913483,0.529370,1.781133,1.781114,1.787648,0.872110,1.405889,0.371831,0.134294,4.832783e+09,6.040978e+10
2,3,Total Cen_01126,Ano 11,90.500000,90.174558,67.648990,99.984848,85.874082,0.912952,0.528184,1.697311,1.697292,1.703826,0.840965,1.470267,0.371831,0.140443,4.755497e+09,5.944371e+10
3,4,Total Cen_01128,Ano 9,83.756944,90.230271,70.665404,98.681818,85.254075,0.907076,0.480585,1.827726,1.827695,1.725630,0.733307,1.734699,0.273995,0.512034,3.736570e+09,4.670713e+10
4,5,Total Cen_01127,Ano 9,83.687500,90.183081,70.622475,96.795455,84.869605,0.906578,0.479142,1.746243,1.746212,1.644148,0.711007,1.804504,0.273995,0.532639,3.680666e+09,4.600833e+10
5,6,Total Cen_01128,Ano 10,87.565972,86.483112,69.607323,99.931818,84.522727,0.912129,0.504078,1.627058,1.627036,1.590875,0.764697,1.566764,0.343711,0.291636,4.264055e+09,5.330068e+10
6,7,Total Cen_01125,Ano 11,89.722222,86.481218,67.643939,99.977273,84.386880,0.912414,0.526984,1.614145,1.614126,1.620660,0.810063,1.540246,0.371831,0.147128,4.679023e+09,5.848779e+10
7,8,Total Cen_01127,Ano 10,87.552083,86.376263,68.993687,99.909091,84.309860,0.911631,0.502733,1.551463,1.551441,1.515280,0.738737,1.633190,0.343711,0.304001,4.197702e+09,5.247128e+10
8,9,Total Cen_01124,Ano 11,88.750000,86.358270,67.636364,99.969697,84.161961,0.911869,0.525770,1.531629,1.531610,1.538144,0.779404,1.616586,0.371831,0.154420,4.603354e+09,5.754192e+10
9,10,Total Cen_01126,Ano 10,87.527778,86.293245,68.233586,99.886364,84.063820,0.911127,0.501373,1.476406,1.476384,1.440223,0.712962,1.704960,0.343711,0.317360,4.131984e+09,5.164980e+10


## Análise Comparativa dos Resultados

Após a construção do ranking, os indicadores são analisados ao longo dos anos para compreender melhor o comportamento financeiro dos cenários simulados.

Nesta etapa, os resultados dos 1.200 cenários são consolidados por ano, permitindo observar tendências gerais de rentabilidade, liquidez, estrutura de capital e geração de valor.

A comparação também ajuda a interpretar o resultado do ranking, verificando se os anos que aparecem entre as melhores posições apresentam características financeiras que se destacam em relação aos demais períodos.

O Ano 12 é mantido apenas nos indicadores em que existem dados adequados para análise. Nos casos em que a estrutura contábil desse período não permite uma comparação equivalente, o valor permanece indisponível.

In [11]:
ordem_anos = [f"Ano {i}" for i in range(1, 13)]

indicadores_comparacao = [
    "Margem EBITDA",
    "Margem Líquida",
    "LC",
    "LS",
    "LI",
    "LG",
    "PCT",
    "CE",
    "IPL",
    "NOPAT",
    "Capital Investido",
    "EVA",
    "MVA Aproximado"
]

comparativo_anual = (
    kpis_final
    .groupby("Ano", observed=False)[indicadores_comparacao]
    .median()
    .reindex(ordem_anos)
    .reset_index()
)

display(comparativo_anual)

,Ano,Margem EBITDA,Margem Líquida,LC,LS,LI,LG,PCT,CE,IPL,NOPAT,Capital Investido,EVA,MVA Aproximado
0,Ano 1,0.829471,0.391252,1.651016,1.650936,1.101716,0.281286,4.309074,0.125421,4.097232,2.784703e+09,6.735976e+09,2.245657e+09,2.807071e+10
1,Ano 2,0.825971,0.355374,1.140324,1.140260,0.735255,0.275336,6.213279,0.152360,5.503329,2.315685e+09,6.412141e+09,1.802821e+09,2.253527e+10
2,Ano 3,0.855565,0.371963,1.298966,1.298917,0.986973,0.374552,6.223049,0.188947,4.892188,2.440991e+09,5.778747e+09,1.978748e+09,2.473436e+10
3,Ano 4,0.868126,0.390422,3.377675,3.377494,2.267617,0.374361,5.042551,0.060902,4.154821,2.572875e+09,6.981495e+09,2.014212e+09,2.517765e+10
4,Ano 5,0.875065,0.423980,0.668341,0.668272,0.229612,0.248062,4.939364,0.144637,3.279878,2.835020e+09,9.179809e+09,2.100693e+09,2.625867e+10
5,Ano 6,0.881416,0.446896,0.401450,0.401395,0.129781,0.249278,4.812715,0.211153,2.576103,3.061159e+09,1.027142e+10,2.239483e+09,2.799354e+10
6,Ano 7,0.887740,0.449379,0.377550,0.377516,0.169773,0.284343,4.672226,0.249263,2.006060,3.222436e+09,1.113087e+10,2.331956e+09,2.914945e+10
7,Ano 8,0.891765,0.456720,0.740137,0.740101,0.565999,0.405669,4.195971,0.239410,1.470935,3.466271e+09,1.073262e+10,2.607680e+09,3.259600e+10
8,Ano 9,0.895466,0.467417,0.758536,0.758506,0.654316,0.440571,3.981239,0.270643,1.055790,3.698462e+09,1.081029e+10,2.833715e+09,3.542144e+10
9,Ano 10,0.900470,0.496231,0.581804,0.581782,0.545712,0.405727,3.874671,0.341930,0.647875,4.090598e+09,1.123286e+10,3.191907e+09,3.989884e+10


### Evolução da Rentabilidade

A evolução das margens permite comparar o desempenho operacional e o resultado líquido ao longo dos anos.

A análise busca identificar se os períodos que aparecem com maior frequência entre as melhores posições do ranking também apresentam níveis superiores de rentabilidade em relação aos demais anos.

In [ ]:
import matplotlib.pyplot as plt

COR_EBITDA = "#7D6B9D"
COR_LIQUIDA = "#C27C8E"

COR_TEXTO = "#504C57"
COR_GRADE = "#DDD8E3"
COR_FUNDO = "#FAF9FB"

fig, ax = plt.subplots(figsize=(12, 6))

fig.patch.set_facecolor(COR_FUNDO)
ax.set_facecolor(COR_FUNDO)

ax.plot(
    comparativo_anual["Ano"],
    comparativo_anual["Margem EBITDA"] * 100,
    marker="o",
    linewidth=2.5,
    color=COR_EBITDA,
    label="Margem EBITDA"
)

ax.plot(
    comparativo_anual["Ano"],
    comparativo_anual["Margem Líquida"] * 100,
    marker="o",
    linewidth=2.5,
    color=COR_LIQUIDA,
    label="Margem Líquida"
)

ax.set_title(
    "Evolução da Rentabilidade",
    fontsize=16,
    fontweight="bold",
    color=COR_TEXTO,
    pad=18
)

ax.set_ylabel("Margem (%)", color=COR_TEXTO)
ax.set_xlabel("Ano", color=COR_TEXTO)

ax.yaxis.grid(
    True,
    linestyle="--",
    alpha=0.4,
    color=COR_GRADE
)

ax.set_axisbelow(True)

for lado in ["top", "right", "left"]:
    ax.spines[lado].set_visible(False)

ax.spines["bottom"].set_color(COR_GRADE)

ax.tick_params(
    axis="both",
    colors=COR_TEXTO,
    length=0
)

ax.legend(frameon=False)

plt.tight_layout()
plt.show()

### Evolução da Liquidez

Os indicadores de liquidez são comparados ao longo dos anos para avaliar a capacidade financeira de curto e longo prazo.

A análise permite observar quais períodos apresentaram condições mais favoráveis e verificar como o comportamento da liquidez contribuiu para a pontuação dos anos que tiveram maior destaque no ranking.

In [ ]:
CORES_LIQUIDEZ = {
    "LC": "#557FA3",
    "LS": "#78A6C8",
    "LI": "#72A58A",
    "LG": "#C99568"
}

COR_TEXTO = "#504C57"
COR_GRADE = "#DDD8E3"
COR_FUNDO = "#FAF9FB"

fig, ax = plt.subplots(figsize=(12, 6))

fig.patch.set_facecolor(COR_FUNDO)
ax.set_facecolor(COR_FUNDO)

for indicador in ["LC", "LS", "LI", "LG"]:
    ax.plot(
        comparativo_anual["Ano"],
        comparativo_anual[indicador],
        marker="o",
        linewidth=2.2,
        color=CORES_LIQUIDEZ[indicador],
        label=indicador
    )

ax.axhline(
    y=1,
    linestyle="--",
    linewidth=1.2,
    color="#A9A5AD",
    alpha=0.8
)

ax.set_title(
    "Evolução dos Indicadores de Liquidez",
    fontsize=16,
    fontweight="bold",
    color=COR_TEXTO,
    pad=18
)

ax.set_ylabel("Índice", color=COR_TEXTO)
ax.set_xlabel("Ano", color=COR_TEXTO)

ax.yaxis.grid(
    True,
    linestyle="--",
    alpha=0.35,
    color=COR_GRADE
)

ax.set_axisbelow(True)

for lado in ["top", "right", "left"]:
    ax.spines[lado].set_visible(False)

ax.spines["bottom"].set_color(COR_GRADE)

ax.tick_params(
    axis="both",
    colors=COR_TEXTO,
    length=0
)

ax.legend(
    frameon=False,
    ncol=4
)

plt.tight_layout()
plt.show()

### Evolução da Estrutura de Capital

A estrutura de capital é analisada por meio da Participação de Capital de Terceiros, da Composição do Endividamento e da Imobilização do Patrimônio Líquido.

A comparação ao longo dos anos permite observar mudanças no perfil de endividamento e avaliar como essas mudanças influenciaram o desempenho financeiro.

In [ ]:
COR_PCT = "#5E7896"
COR_IPL = "#A06F83"
COR_CE = "#D09A58"

COR_TEXTO = "#504C57"
COR_GRADE = "#DDD8E3"
COR_FUNDO = "#FAF9FB"

fig, ax1 = plt.subplots(figsize=(12, 6))

fig.patch.set_facecolor(COR_FUNDO)
ax1.set_facecolor(COR_FUNDO)

linha_pct, = ax1.plot(
    comparativo_anual["Ano"],
    comparativo_anual["PCT"],
    marker="o",
    linewidth=2.4,
    color=COR_PCT,
    label="PCT"
)

linha_ipl, = ax1.plot(
    comparativo_anual["Ano"],
    comparativo_anual["IPL"],
    marker="o",
    linewidth=2.4,
    color=COR_IPL,
    label="IPL"
)

ax2 = ax1.twinx()

linha_ce, = ax2.plot(
    comparativo_anual["Ano"],
    comparativo_anual["CE"] * 100,
    marker="o",
    linewidth=2.4,
    linestyle="--",
    color=COR_CE,
    label="CE"
)

ax1.set_title(
    "Evolução da Estrutura de Capital",
    fontsize=16,
    fontweight="bold",
    color=COR_TEXTO,
    pad=18
)

ax1.set_xlabel("Ano", color=COR_TEXTO)
ax1.set_ylabel("PCT / IPL", color=COR_TEXTO)
ax2.set_ylabel("CE (%)", color=COR_TEXTO)

ax1.yaxis.grid(
    True,
    linestyle="--",
    alpha=0.35,
    color=COR_GRADE
)

ax1.set_axisbelow(True)

for lado in ["top", "left"]:
    ax1.spines[lado].set_visible(False)

ax2.spines["top"].set_visible(False)

ax1.tick_params(axis="both", colors=COR_TEXTO, length=0)
ax2.tick_params(axis="y", colors=COR_TEXTO, length=0)

ax1.legend(
    [linha_pct, linha_ipl, linha_ce],
    ["PCT", "IPL", "CE"],
    frameon=False,
    ncol=3
)

plt.tight_layout()
plt.show()

### Evolução da Geração de Valor

A geração de valor é analisada a partir do NOPAT, do Capital Investido e do EVA.

A comparação permite observar como o resultado operacional após impostos evoluiu em relação ao capital necessário para sustentar a operação e ao valor econômico efetivamente gerado.

Essa análise complementa o ranking ao mostrar se os anos mais bem posicionados também apresentam maior capacidade de geração de valor econômico.

In [ ]:
COR_NOPAT = "#527F91"
COR_CAPITAL = "#B7A7C8"
COR_EVA = "#65977B"

COR_TEXTO = "#504C57"
COR_GRADE = "#DDD8E3"
COR_FUNDO = "#FAF9FB"

fig, ax = plt.subplots(figsize=(12, 6))

fig.patch.set_facecolor(COR_FUNDO)
ax.set_facecolor(COR_FUNDO)

anos = comparativo_anual["Ano"]

ax.bar(
    anos,
    comparativo_anual["Capital Investido"] / 1_000_000_000,
    color=COR_CAPITAL,
    alpha=0.75,
    label="Capital Investido"
)

ax.plot(
    anos,
    comparativo_anual["NOPAT"] / 1_000_000_000,
    marker="o",
    linewidth=2.5,
    color=COR_NOPAT,
    label="NOPAT"
)

ax.plot(
    anos,
    comparativo_anual["EVA"] / 1_000_000_000,
    marker="o",
    linewidth=2.5,
    color=COR_EVA,
    label="EVA"
)

ax.set_title(
    "Evolução da Geração de Valor",
    fontsize=16,
    fontweight="bold",
    color=COR_TEXTO,
    pad=18
)

ax.set_xlabel("Ano", color=COR_TEXTO)
ax.set_ylabel("R$ bilhões", color=COR_TEXTO)

ax.yaxis.grid(
    True,
    linestyle="--",
    alpha=0.35,
    color=COR_GRADE
)

ax.set_axisbelow(True)

for lado in ["top", "right", "left"]:
    ax.spines[lado].set_visible(False)

ax.spines["bottom"].set_color(COR_GRADE)

ax.tick_params(
    axis="both",
    colors=COR_TEXTO,
    length=0
)

ax.legend(
    frameon=False,
    ncol=3
)

plt.tight_layout()
plt.show()

## Exportação do Ranking 

Nesta etapa, os resultados finais são adicionados à planilha financeira.

A aba **Ranking** apresenta as combinações de cenário e ano que obtiveram as maiores pontuações, considerando conjuntamente rentabilidade, liquidez, estrutura de capital e geração de valor.
g.

## Exportação da Análise Financeira

Os resultados calculados são organizados em uma planilha Excel para facilitar a consulta e a análise dos indicadores.

In [12]:
from pathlib import Path
import pandas as pd
import numpy as np

from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.utils import get_column_letter
from openpyxl.workbook.defined_name import DefinedName


# Caminho de saída

# Entrega 2: a planilha é gerada na pasta da Entrega 2 (a Entrega 1 não é alterada).
# Se a pasta ainda não existir, ela é criada.
pasta_saida = Path("../documentos/entrega 2/contabilidade e finanças")
pasta_saida.mkdir(parents=True, exist_ok=True)

arquivo_saida = pasta_saida / "Analise_Financeira.xlsx"


# Preparação da base

base = df_kpis[
    ["Cenario", "Ano", "Conta", "Valor_numerico"]
].copy()

base["Valor_numerico"] = base["Valor_numerico"] / 1000

base.columns = [
    "Cenário",
    "Ano",
    "Conta",
    "Valor (R$ mil)"
]


# Preparação dos indicadores

indicadores = kpis_final.copy()

# Entrega 2: confere se os novos indicadores chegaram até aqui
faltando = [c for c in colunas_novos_indicadores if c not in indicadores.columns]

if faltando:
    raise KeyError(
        f"Indicadores ausentes no kpis_final: {faltando}. "
        "Execute as células 'Consolidação dos KPIs' e 'Consolidação dos Indicadores', nessa ordem."
    )

# Valores em reais passam para R$ mil (a Entrega 2 inclui CCL, FCO e FCL)
for col in [
    "NOPAT",
    "Capital Investido",
    "EVA",
    "MVA Aproximado",
    "CCL",
    "Necessidade de Capital de Giro",
    "Recebimentos Estimados",
    "FCO",
    "FCL"
]:
    indicadores[col] = indicadores[col] / 1000

# Divisões por zero não podem ir para o Excel
indicadores = indicadores.replace([np.inf, -np.inf], np.nan)


# EBITDA

ebitda_base = (
    df_kpis[
        df_kpis["Conta"] == "DRE - EBITDA"
    ][["Cenario", "Ano", "Valor_numerico"]]
    .copy()
)

ebitda_base["EBITDA"] = ebitda_base["Valor_numerico"] / 1000

ebitda_base = ebitda_base[
    ["Cenario", "Ano", "EBITDA"]
]

indicadores = indicadores.merge(
    ebitda_base,
    on=["Cenario", "Ano"],
    how="left"
)


# Cenários e anos

cenarios = sorted(
    indicadores["Cenario"].unique()
)

anos = [
    f"Ano {i}"
    for i in range(1, 13)
]


# Dicionário de KPIs

dicionario = pd.DataFrame([
    [
        "Margem EBITDA",
        "EBITDA / Receita Líquida",
        "%",
        "Indica quanto da receita líquida é convertida em EBITDA."
    ],
    [
        "Margem Líquida",
        "Resultado Líquido / Receita Líquida",
        "%",
        "Indica quanto da receita líquida permanece como resultado líquido."
    ],
    [
        "Liquidez Corrente (LC)",
        "Ativo Circulante / Passivo Circulante",
        "Índice",
        "Avalia a capacidade de pagamento das obrigações de curto prazo."
    ],
    [
        "Liquidez Seca (LS)",
        "(Ativo Circulante - Estoques) / Passivo Circulante",
        "Índice",
        "Avalia a liquidez de curto prazo sem considerar os estoques."
    ],
    [
        "Liquidez Imediata (LI)",
        "Disponível / Passivo Circulante",
        "Índice",
        "Avalia a capacidade de pagamento imediato das obrigações de curto prazo."
    ],
    [
        "Liquidez Geral (LG)",
        "(AC + RLP) / (PC + ELP)",
        "Índice",
        "Avalia a capacidade de pagamento considerando curto e longo prazo."
    ],
    [
        "Participação de Capital de Terceiros (PCT)",
        "(PC + ELP) / Patrimônio Líquido",
        "Índice",
        "Mostra a relação entre recursos de terceiros e patrimônio líquido."
    ],
    [
        "Composição do Endividamento (CE)",
        "PC / (PC + ELP)",
        "%",
        "Mostra a parcela do endividamento concentrada no curto prazo."
    ],
    [
        "Imobilização do Patrimônio Líquido (IPL)",
        "Permanente / Patrimônio Líquido",
        "Índice",
        "Indica quanto do patrimônio líquido está aplicado no permanente."
    ],
    [
        "EBITDA",
        "Valor disponibilizado na base",
        "R$ mil",
        "Resultado antes de juros, impostos, depreciação e amortização."
    ],
    [
        "NOPAT",
        "Resultado Operacional - Imposto estimado sobre o Resultado Operacional",
        "R$ mil",
        "Resultado operacional após a estimativa dos impostos."
    ],
    [
        "Capital Investido",
        "Patrimônio Líquido + Dívidas Financeiras - Disponível",
        "R$ mil",
        "Capital utilizado como base para o cálculo da criação de valor."
    ],
    [
        "EVA",
        "NOPAT - (Capital Investido × WACC)",
        "R$ mil",
        "Indica o valor econômico criado após remunerar o capital investido."
    ],
    [
        "MVA Aproximado",
        "EVA / WACC",
        "R$ mil",
        "Estimativa do valor agregado baseada na perpetuidade do EVA."
    ],
    [
        "ROIC",
        "NOPAT / Capital Investido",
        "%",
        "Indicador auxiliar utilizado na análise do retorno sobre o capital investido."
    ],
    [
        "WACC",
        "Custo médio ponderado de capital",
        "%",
        "Parâmetro auxiliar de 8% utilizado nos cálculos de EVA e MVA."
    ]
], columns=[
    "Indicador",
    "Fórmula / critério",
    "Unidade",
    "Interpretação"
])


# Entrega 2: novos indicadores no dicionário

novos_indicadores = pd.DataFrame([
    [
        "ROE",
        "Resultado Líquido / Patrimônio Líquido Médio",
        "%",
        "Mede o retorno obtido sobre o capital próprio dos acionistas."
    ],
    [
        "DuPont - Margem Líquida",
        "Resultado Líquido / Receita Líquida",
        "%",
        "Mostra quanto da receita líquida permanece como resultado."
    ],
    [
        "DuPont - Giro do Ativo",
        "Receita Líquida / Ativo Total Médio",
        "Índice",
        "Mede quanto de receita é gerado por unidade de ativo."
    ],
    [
        "DuPont - Alavancagem",
        "Ativo Total Médio / Patrimônio Líquido Médio",
        "Índice",
        "Mostra a relação entre os ativos e o capital próprio."
    ],
    [
        "ROE DuPont",
        "Margem Líquida × Giro do Ativo × Alavancagem",
        "%",
        "Decompõe o retorno sobre o patrimônio líquido em três fatores."
    ],
    [
        "CCL",
        "Ativo Circulante − Passivo Circulante",
        "R$ mil",
        "Indica a folga financeira de curto prazo, usando o passivo informado."
    ],
    [
        "PMR",
        "Contas a Receber de Clientes / Receita Operacional × 360",
        "Dias",
        "Estima o prazo médio de recebimento das vendas."
    ],
    [
        "PMP",
        "Fornecedores / Custos × 360",
        "Dias",
        "Estima o prazo médio de pagamento aos fornecedores."
    ],
    [
        "Ciclo Financeiro",
        "PMR − PMP",
        "Dias",
        "Indica o intervalo entre pagar fornecedores e receber dos clientes."
    ],
    [
        "Necessidade de Capital de Giro",
        "Ciclo Financeiro × Custos / 360",
        "R$ mil",
        "Estimativa didática do capital necessário para financiar o ciclo."
    ],
    [
        "Taxa de Conversão da Receita em Caixa",
        "Recebimentos Estimados / Receita",
        "%",
        "Estima quanto da receita do período se converteu em recebimentos."
    ],
    [
        "FCO",
        "Recebimentos − Tributos pagos − Custos pagos − IR/CS pago",
        "R$ mil",
        "Estima o fluxo de caixa operacional com as contas disponíveis."
    ],
    [
        "FCO / EBITDA",
        "FCO / EBITDA",
        "%",
        "Mede quanto do EBITDA se converte em caixa operacional."
    ],
    [
        "FCO / Lucro Líquido",
        "FCO / Lucro Líquido",
        "Índice",
        "Avalia a conversão do lucro contábil em caixa."
    ],
    [
        "FCL",
        "FCO − Investimentos",
        "R$ mil",
        "Estima o caixa livre após os investimentos."
    ],
    [
        "Capex / Receita",
        "Investimentos / Receita",
        "%",
        "Mede a parcela da receita destinada a investimentos."
    ],
    [
        "Capex / D&A",
        "Investimentos / Depreciação e Amortização",
        "Índice",
        "Compara o investimento com o desgaste dos ativos."
    ],
    [
        "Cobertura de Juros",
        "FCO / Juros pagos",
        "Índice",
        "Mede a capacidade do caixa operacional de cobrir os juros."
    ],
    [
        "FCO / Dívida Bruta",
        "FCO / (Empréstimos CP + LP)",
        "%",
        "Compara o caixa operacional com o saldo de dívidas financeiras."
    ],
    [
        "Payout em Caixa",
        "Distribuição a Acionistas / Lucro Líquido",
        "%",
        "Compara a distribuição informada aos acionistas com o lucro."
    ],
    [
        "Cobertura de Dividendos",
        "FCL / Distribuição a Acionistas",
        "Índice",
        "Avalia se o caixa livre cobre as distribuições informadas."
    ]
], columns=dicionario.columns)

dicionario = pd.concat(
    [dicionario, novos_indicadores],
    ignore_index=True
)


# Gravação inicial

print("Gravando dados...")

with pd.ExcelWriter(
    arquivo_saida,
    engine="xlsxwriter"
) as writer:

    dicionario.to_excel(
        writer,
        sheet_name="Dicionário de KPIs financeiros",
        index=False,
        startrow=3
    )

    base.to_excel(
        writer,
        sheet_name="Dados de base",
        index=False,
        startrow=3
    )

    indicadores.to_excel(
        writer,
        sheet_name="_Indicadores",
        index=False
    )


# Aplicação do layout

print("Aplicando layout...")

wb = load_workbook(arquivo_saida)

ws_dic = wb["Dicionário de KPIs financeiros"]
ws_base = wb["Dados de base"]
ws_aux = wb["_Indicadores"]

ws_demo = wb.create_sheet("Demonstrações contábeis")
ws_ind = wb.create_sheet("Indicadores - Índices")
ws_rank = wb.create_sheet("Ranking")
ws_comp = wb.create_sheet("Análise Comparativa")
ws_listas = wb.create_sheet("_Listas")


# Paleta

GRAFITE = "403D46"
TEXTO = "504C57"

LAVANDA_ESCURO = "756C86"
LAVANDA = "AAA1BA"
LAVANDA_MEDIA = "D7D1E0"
LAVANDA_CLARA = "F1EEF5"

CINZA = "F8F7F9"
BRANCO = "FFFFFF"
BORDA = "DDD8E3"

VERDE = "E2F0E5"
VERDE_TEXTO = "263D2D"

VERMELHO = "F6E3E3"
VERMELHO_TEXTO = "573436"


fill_lavanda_escuro = PatternFill(
    "solid",
    fgColor=LAVANDA_ESCURO
)

fill_lavanda = PatternFill(
    "solid",
    fgColor=LAVANDA
)

fill_lavanda_media = PatternFill(
    "solid",
    fgColor=LAVANDA_MEDIA
)

fill_lavanda_clara = PatternFill(
    "solid",
    fgColor=LAVANDA_CLARA
)

fill_cinza = PatternFill(
    "solid",
    fgColor=CINZA
)

fill_verde = PatternFill(
    "solid",
    fgColor=VERDE
)

fill_vermelho = PatternFill(
    "solid",
    fgColor=VERMELHO
)

borda_fina = Side(
    style="thin",
    color=BORDA
)


# Funções de formatação

def titulo(ws, texto, subtitulo):

    ws.sheet_view.showGridLines = False

    ws.merge_cells("A1:H1")
    ws["A1"] = texto

    ws["A1"].font = Font(
        name="Aptos Display",
        size=20,
        bold=True,
        color=GRAFITE
    )

    ws["A1"].alignment = Alignment(
        vertical="center"
    )

    ws.row_dimensions[1].height = 32

    ws.merge_cells("A2:H2")
    ws["A2"] = subtitulo

    ws["A2"].font = Font(
        name="Aptos",
        size=10,
        color=TEXTO
    )


def cabecalho(ws, linha, inicio, fim):

    for col in range(inicio, fim + 1):

        c = ws.cell(linha, col)

        c.fill = fill_lavanda_escuro

        c.font = Font(
            name="Aptos",
            size=10,
            bold=True,
            color=BRANCO
        )

        c.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True
        )

        c.border = Border(
            bottom=borda_fina
        )


def seletor(ws):

    ws["A4"] = "CENÁRIO"

    ws["A4"].font = Font(
        name="Aptos",
        size=9,
        bold=True,
        color=LAVANDA_ESCURO
    )

    ws["B4"] = cenarios[0]

    ws["B4"].fill = fill_lavanda_clara

    ws["B4"].font = Font(
        name="Aptos",
        size=10,
        bold=True,
        color=GRAFITE
    )

    ws["B4"].alignment = Alignment(
        horizontal="center"
    )

    ws["B4"].border = Border(
        left=borda_fina,
        right=borda_fina,
        top=borda_fina,
        bottom=borda_fina
    )

    dv = DataValidation(
        type="list",
        formula1="=ListaCenarios",
        allow_blank=False
    )

    ws.add_data_validation(dv)
    dv.add(ws["B4"])


# Lista de cenários

for i, cenario in enumerate(cenarios, 1):

    ws_listas.cell(
        i,
        1,
        cenario
    )


wb.defined_names.add(
    DefinedName(
        "ListaCenarios",
        attr_text=f"'_Listas'!$A$1:$A${len(cenarios)}"
    )
)


# Dicionário

titulo(
    ws_dic,
    "Dicionário de KPIs financeiros",
    "Definições, fórmulas e interpretação dos indicadores utilizados."
)

cabecalho(
    ws_dic,
    4,
    1,
    4
)

ws_dic.column_dimensions["A"].width = 40
ws_dic.column_dimensions["B"].width = 58
ws_dic.column_dimensions["C"].width = 15
ws_dic.column_dimensions["D"].width = 70


for linha in range(
    5,
    ws_dic.max_row + 1
):

    ws_dic.row_dimensions[linha].height = 40

    for col in range(1, 5):

        c = ws_dic.cell(
            linha,
            col
        )

        c.alignment = Alignment(
            vertical="center",
            wrap_text=True
        )

        c.border = Border(
            bottom=borda_fina
        )

        if linha % 2 == 0:
            c.fill = fill_lavanda_clara


ws_dic.freeze_panes = "A5"


# Demonstrações contábeis

titulo(
    ws_demo,
    "Demonstrações contábeis",
    "Selecione um cenário para visualizar BP, DRE e fluxo de caixa."
)

seletor(ws_demo)


ordem_contas = [
    "BAL - Ativo Circulante",
    "BAL - Disponível",
    "BAL - Contas a Receber - Clientes",
    "BAL - Contas a Receber - Partes Relacionadas",
    "BAL - Contas a Receber - SWAP",
    "BAL - Estoques Diversos",
    "BAL - Créditos Tributários",
    "BAL - Outros Créditos",
    "BAL - Realizável a Longo Prazo",
    "BAL - Outros Créditos LP",
    "BAL - At Fiscal Diferido",
    "BAL - Investimentos - Imobilizado",
    "BAL - Investimentos - Intangível",
    "BAL - Outorga da Concessão",
    "BAL - Diferido",
    "BAL - Amortização - Intangível",
    "BAL - Amortização Acumulada",
    "BAL - Depreciação Acumulada",
    "BAL - Permanente",
    "BAL - Total do Ativo",
    "BAL - Passivo Circulante",
    "BAL - Fornecedores",
    "BAL - Contas a Pagar - Parte Relacionada",
    "BAL - Emprést",
    "BAL - Empréstimos",
    "BAL - Tributos a pagar",
    "BAL - Impostos",
    "BAL - Encargos Sociais e Trabalhistas",
    "BAL - Obrigações com o Poder Concedente",
    "BAL - Provisão Manutenção",
    "BAL - Prov para Contingências",
    "BAL - Outros Débitos",
    "BAL - Outros deb",
    "BAL - Exigível a Longo Prazo",
    "BAL - Total do Passivo",
    "BAL - Patrimônio Líquido",
    "BAL - Capital Social",
    "BAL - Reservas Legais",
    "BAL - Reservas de Capital",
    "BAL - Reserva de Retenção de Lucros",
    "BAL - Resultado Acumulado",
    "BAL - Resultado do Período",
    "BAL - Dividendos Antecipados",

    "DRE - Receita",
    "DRE - Tributos",
    "DRE - Custos",
    "DRE - Depreciação e Amortização",
    "DRE - Resultado Operacional",
    "DRE - Outros Resultados Operacionais",
    "DRE - Resultado Financeiro",
    "DRE - Resultado Antes do Imposto de Renda",
    "DRE - Imposto de Renda e Contribuição Social",
    "DRE - Resultado Líquido",
    "DRE - Resultado Líquido após Equivalência",
    "DRE - Receitas Financeiras",
    "DRE - Despesas Financeiras",
    "DRE - EBITDA",

    "FLU - Entradas",
    "FLU - Receita",
    "FLU - Tributos",
    "FLU - Custos",
    "FLU - Resultado Financeiro",
    "FLU - Despesas Financeiras",
    "FLU - Imposto de Renda e Contribuição Social",
    "FLU - Geração de Caixa",
    "FLU - Investimentos",
    "FLU - Distribuição para Acionista",
    "FLU - Saldo Inicial",
    "FLU - Saldo Final"
]


ws_demo["A6"] = "Conta"

for col, ano in enumerate(
    anos,
    2
):

    ws_demo.cell(
        6,
        col,
        f"{ano}\n(R$ mil)"
    )


cabecalho(
    ws_demo,
    6,
    1,
    13
)

ws_demo.row_dimensions[6].height = 40


linha = 7

secoes = [
    ("BALANÇO PATRIMONIAL", "BAL -"),
    ("DEMONSTRAÇÃO DO RESULTADO", "DRE -"),
    ("FLUXO DE CAIXA", "FLU -")
]


for nome_secao, prefixo in secoes:

    ws_demo.merge_cells(
        start_row=linha,
        start_column=1,
        end_row=linha,
        end_column=13
    )

    c = ws_demo.cell(
        linha,
        1
    )

    c.value = nome_secao
    c.fill = fill_lavanda

    c.font = Font(
        name="Aptos",
        size=10,
        bold=True,
        color=GRAFITE
    )

    ws_demo.row_dimensions[linha].height = 24

    linha += 1


    contas_secao = [
        conta
        for conta in ordem_contas
        if conta.startswith(prefixo)
    ]


    for conta in contas_secao:

        ws_demo.cell(
            linha,
            1,
            conta.replace(prefixo, "")
        )


        for col, ano in enumerate(
            anos,
            2
        ):

            formula = (
                f'=IF('
                f'COUNTIFS('
                f"'Dados de base'!$A:$A,$B$4,"
                f"'Dados de base'!$B:$B,\"{ano}\","
                f"'Dados de base'!$C:$C,\"{conta}\""
                f')=0,'
                f'"N/D",'
                f'SUMIFS('
                f"'Dados de base'!$D:$D,"
                f"'Dados de base'!$A:$A,$B$4,"
                f"'Dados de base'!$B:$B,\"{ano}\","
                f"'Dados de base'!$C:$C,\"{conta}\""
                f'))'
            )

            ws_demo.cell(
                linha,
                col
            ).value = formula

            ws_demo.cell(
                linha,
                col
            ).number_format = '#,##0.00'


        if any(
            x in conta
            for x in [
                "Total do Ativo",
                "Total do Passivo",
                "Patrimônio Líquido",
                "Resultado Líquido",
                "EBITDA",
                "Geração de Caixa",
                "Saldo Final"
            ]
        ):

            for col in range(
                1,
                14
            ):

                ws_demo.cell(
                    linha,
                    col
                ).fill = fill_lavanda_clara

                ws_demo.cell(
                    linha,
                    col
                ).font = Font(
                    name="Aptos",
                    size=10,
                    bold=True,
                    color=GRAFITE
                )


        linha += 1


ws_demo.column_dimensions["A"].width = 43

for col in range(
    2,
    14
):

    ws_demo.column_dimensions[
        get_column_letter(col)
    ].width = 16


ws_demo.freeze_panes = "B7"


# Indicadores

titulo(
    ws_ind,
    "Indicadores / Índices financeiros",
    "Selecione um cenário para acompanhar os indicadores ao longo dos 12 anos."
)

seletor(ws_ind)

ws_ind["A6"] = "Indicador"


for col, ano in enumerate(
    anos,
    2
):

    ws_ind.cell(
        6,
        col,
        ano
    )


cabecalho(
    ws_ind,
    6,
    1,
    13
)

ws_ind.row_dimensions[6].height = 34


estrutura_indicadores = [
    ("Margem EBITDA (%)", "Margem EBITDA", "percentual"),
    ("Margem Líquida (%)", "Margem Líquida", "percentual"),
    ("Liquidez Corrente (LC)", "LC", "indice"),
    ("Liquidez Seca (LS)", "LS", "indice"),
    ("Liquidez Imediata (LI)", "LI", "indice"),
    ("Liquidez Geral (LG)", "LG", "indice"),
    (
        "Participação de Capital de Terceiros (PCT)",
        "PCT",
        "indice"
    ),
    (
        "Composição do Endividamento (CE)",
        "CE",
        "percentual"
    ),
    (
        "Imobilização do Patrimônio Líquido (IPL)",
        "IPL",
        "indice"
    ),
    ("EBITDA (R$ mil)", "EBITDA", "moeda"),
    ("NOPAT (R$ mil)", "NOPAT", "moeda"),
    (
        "Capital Investido (R$ mil)",
        "Capital Investido",
        "moeda"
    ),
    ("EVA (R$ mil)", "EVA", "moeda"),
    (
        "MVA Aproximado (R$ mil)",
        "MVA Aproximado",
        "moeda"
    ),

    # Novos indicadores - Entrega 2
    ("ROE (%)", "ROE", "percentual"),
    ("DuPont - Margem Líquida (%)", "DuPont - Margem Líquida", "percentual"),
    ("DuPont - Giro do Ativo", "DuPont - Giro do Ativo", "indice"),
    ("DuPont - Alavancagem", "DuPont - Alavancagem", "indice"),
    ("ROE DuPont (%)", "ROE DuPont", "percentual"),

    ("CCL (R$ mil)", "CCL", "moeda"),
    ("PMR (dias)", "PMR", "indice"),
    ("PMP (dias)", "PMP", "indice"),
    ("Ciclo Financeiro (dias)", "Ciclo Financeiro", "indice"),
    (
        "Necessidade de Capital de Giro (R$ mil)",
        "Necessidade de Capital de Giro",
        "moeda"
    ),

    (
        "Taxa de Conversão da Receita em Caixa (%)",
        "Taxa de Conversão da Receita em Caixa",
        "percentual"
    ),
    ("FCO (R$ mil)", "FCO", "moeda"),
    ("FCO / EBITDA (%)", "FCO / EBITDA", "percentual"),
    ("FCO / Lucro Líquido", "FCO / Lucro Líquido", "indice"),
    ("FCL (R$ mil)", "FCL", "moeda"),

    ("Capex / Receita (%)", "Capex / Receita", "percentual"),
    ("Capex / D&A", "Capex / D&A", "indice"),
    ("Cobertura de Juros", "Cobertura de Juros", "indice"),
    ("FCO / Dívida Bruta (%)", "FCO / Dívida Bruta", "percentual"),
    ("Payout em Caixa (%)", "Payout em Caixa", "percentual"),
    ("Cobertura de Dividendos", "Cobertura de Dividendos", "indice")
]


headers_ind = list(
    indicadores.columns
)

linhas_indicadores = {}


for linha, (
    nome_exibicao,
    nome_coluna,
    tipo
) in enumerate(
    estrutura_indicadores,
    7
):

    linhas_indicadores[
        nome_coluna
    ] = linha

    ws_ind.cell(
        linha,
        1,
        nome_exibicao
    )

    ws_ind.cell(
        linha,
        1
    ).font = Font(
        name="Aptos",
        size=10,
        bold=True,
        color=GRAFITE
    )


    if linha % 2 == 0:

        for col in range(
            1,
            14
        ):

            ws_ind.cell(
                linha,
                col
            ).fill = fill_lavanda_clara


    letra = get_column_letter(
        headers_ind.index(
            nome_coluna
        ) + 1
    )


    for col, ano in enumerate(
        anos,
        2
    ):

        celula = ws_ind.cell(
            linha,
            col
        )

        col_ano = get_column_letter(
            col
        )

        formula_contagem = (
            f'COUNTIFS('
            f"'_Indicadores'!$A:$A,$B$4,"
            f"'_Indicadores'!$B:$B,{col_ano}$6,"
            f"'_Indicadores'!${letra}:${letra},\"<>\""
            f')'
        )

        formula_valor = (
            f'SUMIFS('
            f"'_Indicadores'!${letra}:${letra},"
            f"'_Indicadores'!$A:$A,$B$4,"
            f"'_Indicadores'!$B:$B,{col_ano}$6"
            f')'
        )

        celula.value = (
            f'=IF({formula_contagem}=0,"N/D",{formula_valor})'
        )


        if tipo == "percentual":
            celula.number_format = "0.00%"

        elif tipo == "indice":
            celula.number_format = "0.00"

        elif tipo == "moeda":
            celula.number_format = '#,##0.00'


        celula.alignment = Alignment(
            horizontal="center",
            vertical="center"
        )


# Cores dos indicadores

def aplicar_fundo(
    linha,
    condicao
):

    for col in range(
        2,
        14
    ):

        nome_coluna = None

        for nome, lin in linhas_indicadores.items():

            if lin == linha:
                nome_coluna = nome
                break


        if nome_coluna is None:
            continue


        ano = anos[
            col - 2
        ]

        filtro = indicadores[
            (indicadores["Cenario"] == cenarios[0]) &
            (indicadores["Ano"] == ano)
        ]


        if filtro.empty:
            continue


        valor = filtro[
            nome_coluna
        ].iloc[0]


        if pd.isna(valor):
            continue


        resultado = condicao(
            valor
        )


        if resultado == "ruim":

            ws_ind.cell(
                linha,
                col
            ).fill = fill_vermelho

            ws_ind.cell(
                linha,
                col
            ).font = Font(
                name="Aptos",
                size=10,
                color=VERMELHO_TEXTO
            )


        elif resultado == "bom":

            ws_ind.cell(
                linha,
                col
            ).fill = fill_verde

            ws_ind.cell(
                linha,
                col
            ).font = Font(
                name="Aptos",
                size=10,
                color=VERDE_TEXTO
            )


for nome in [
    "LC",
    "LS",
    "LG"
]:

    aplicar_fundo(
        linhas_indicadores[nome],
        lambda x:
        "ruim" if x < 1
        else "bom" if x >= 1.5
        else None
    )


aplicar_fundo(
    linhas_indicadores["LI"],
    lambda x:
    "ruim" if x < 0.5
    else "bom" if x >= 1
    else None
)


aplicar_fundo(
    linhas_indicadores["PCT"],
    lambda x:
    "ruim" if x >= 4
    else "bom" if x <= 1
    else None
)


aplicar_fundo(
    linhas_indicadores["CE"],
    lambda x:
    "ruim" if x >= 0.6
    else "bom" if x <= 0.3
    else None
)


aplicar_fundo(
    linhas_indicadores["IPL"],
    lambda x:
    "ruim" if x >= 1
    else "bom" if x <= 0.5
    else None
)


aplicar_fundo(
    linhas_indicadores["EVA"],
    lambda x:
    "ruim" if x < 0
    else "bom" if x > 0
    else None
)


aplicar_fundo(
    linhas_indicadores["MVA Aproximado"],
    lambda x:
    "ruim" if x < 0
    else "bom" if x > 0
    else None
)


# Entrega 2: cores dos novos indicadores
# Aqui é usada formatação condicional do Excel, que acompanha o cenário
# escolhido no seletor (as cores são recalculadas quando o cenário muda).

from openpyxl.formatting.rule import FormulaRule

dxf_vermelho = PatternFill(
    start_color=VERMELHO,
    end_color=VERMELHO,
    fill_type="solid"
)

dxf_verde = PatternFill(
    start_color=VERDE,
    end_color=VERDE,
    fill_type="solid"
)


def cor_condicional(nome_coluna, limite_ruim, limite_bom):

    linha = linhas_indicadores[nome_coluna]
    ref = f"B{linha}"
    faixa = f"B{linha}:M{linha}"

    # ISNUMBER evita pintar as células "N/D"
    ws_ind.conditional_formatting.add(
        faixa,
        FormulaRule(
            formula=[f"AND(ISNUMBER({ref}),{ref}<{limite_ruim})"],
            fill=dxf_vermelho,
            font=Font(color=VERMELHO_TEXTO)
        )
    )

    ws_ind.conditional_formatting.add(
        faixa,
        FormulaRule(
            formula=[f"AND(ISNUMBER({ref}),{ref}>={limite_bom})"],
            fill=dxf_verde,
            font=Font(color=VERDE_TEXTO)
        )
    )


# Resultado / caixa: negativo = atenção, positivo = favorável
for nome in ["ROE", "ROE DuPont", "CCL", "FCO", "FCL"]:
    cor_condicional(nome, 0, 0.0000001)

# Coberturas: abaixo de 1 o caixa não cobre a obrigação
for nome in ["Cobertura de Juros", "Cobertura de Dividendos"]:
    cor_condicional(nome, 1, 1)


ws_ind.column_dimensions["A"].width = 45


for col in range(
    2,
    14
):

    ws_ind.column_dimensions[
        get_column_letter(col)
    ].width = 16


for linha in range(
    7,
    7 + len(estrutura_indicadores)
):

    ws_ind.row_dimensions[
        linha
    ].height = 25


ws_ind.freeze_panes = "B7"


# Ranking

titulo(
    ws_rank,
    "Ranking dos Cenários e Anos",
    "Comparação das combinações de cenário e ano com melhor desempenho financeiro."
)


ws_rank.merge_cells("A3:H4")

ws_rank["A3"] = (
    "O ranking reúne as combinações de cenário e ano que apresentaram "
    "os melhores resultados na análise. Como os indicadores possuem escalas "
    "diferentes, eles foram transformados em pontuações comparáveis e "
    "organizados em quatro dimensões: rentabilidade, liquidez, estrutura "
    "de capital e geração de valor. A pontuação final representa o "
    "desempenho conjunto desses fatores."
)

ws_rank["A3"].font = Font(
    name="Aptos",
    size=10,
    color=TEXTO
)

ws_rank["A3"].fill = fill_lavanda_clara

ws_rank["A3"].alignment = Alignment(
    vertical="center",
    wrap_text=True
)

ws_rank.row_dimensions[3].height = 25
ws_rank.row_dimensions[4].height = 25


# Top 10

ws_rank["A6"] = "Top 10 - Resultado Geral"

ws_rank["A6"].font = Font(
    name="Aptos Display",
    size=13,
    bold=True,
    color=GRAFITE
)


colunas_ranking = [
    "Posição",
    "Cenário",
    "Ano",
    "Rentabilidade",
    "Liquidez",
    "Estrutura de Capital",
    "Geração de Valor",
    "Pontuação Final"
]


for col, nome in enumerate(
    colunas_ranking,
    1
):

    ws_rank.cell(
        7,
        col,
        nome
    )


cabecalho(
    ws_rank,
    7,
    1,
    len(colunas_ranking)
)


top10 = ranking_excel.head(10)


for linha_excel, (_, registro) in enumerate(
    top10.iterrows(),
    8
):

    for col_excel, nome_coluna in enumerate(
        colunas_ranking,
        1
    ):

        celula = ws_rank.cell(
            linha_excel,
            col_excel,
            registro[nome_coluna]
        )

        celula.font = Font(
            name="Aptos",
            size=10,
            color=GRAFITE,
            bold=(linha_excel == 8)
        )

        celula.alignment = Alignment(
            horizontal=(
                "left"
                if nome_coluna == "Cenário"
                else "center"
            ),
            vertical="center"
        )

        celula.border = Border(
            bottom=borda_fina
        )


        if linha_excel == 8:
            celula.fill = fill_lavanda_clara

        elif linha_excel % 2 == 0:
            celula.fill = fill_cinza


        if nome_coluna in [
            "Rentabilidade",
            "Liquidez",
            "Estrutura de Capital",
            "Geração de Valor",
            "Pontuação Final"
        ]:
            celula.number_format = "0.00"


# Detalhamento do Top 10

linha_detalhe = 20

ws_rank["A20"] = "Indicadores das melhores posições"

ws_rank["A20"].font = Font(
    name="Aptos Display",
    size=13,
    bold=True,
    color=GRAFITE
)


colunas_detalhe = [
    "Posição",
    "Cenário",
    "Ano",
    "Margem EBITDA",
    "Margem Líquida",
    "LC",
    "LS",
    "LI",
    "LG",
    "PCT",
    "CE",
    "IPL",
    "EVA",
    "MVA Aproximado"
]


for col, nome in enumerate(
    colunas_detalhe,
    1
):

    ws_rank.cell(
        linha_detalhe + 1,
        col,
        nome
    )


cabecalho(
    ws_rank,
    linha_detalhe + 1,
    1,
    len(colunas_detalhe)
)


for linha_excel, (_, registro) in enumerate(
    top10.iterrows(),
    linha_detalhe + 2
):

    for col_excel, nome_coluna in enumerate(
        colunas_detalhe,
        1
    ):

        valor = registro[
            nome_coluna
        ]

        celula = ws_rank.cell(
            linha_excel,
            col_excel,
            valor
        )

        celula.font = Font(
            name="Aptos",
            size=9,
            color=TEXTO
        )

        celula.alignment = Alignment(
            horizontal=(
                "left"
                if nome_coluna == "Cenário"
                else "center"
            ),
            vertical="center"
        )

        celula.border = Border(
            bottom=borda_fina
        )


        if linha_excel % 2 == 0:
            celula.fill = fill_lavanda_clara


        if nome_coluna in [
            "Margem EBITDA",
            "Margem Líquida",
            "CE"
        ]:
            celula.number_format = "0.00%"

        elif nome_coluna in [
            "EVA",
            "MVA Aproximado"
        ]:
            celula.number_format = '#,##0.00'

        elif nome_coluna not in [
            "Posição",
            "Cenário",
            "Ano"
        ]:
            celula.number_format = "0.00"


larguras_rank = {
    "A": 11,
    "B": 22,
    "C": 12,
    "D": 18,
    "E": 18,
    "F": 22,
    "G": 20,
    "H": 18,
    "I": 12,
    "J": 12,
    "K": 12,
    "L": 12,
    "M": 20,
    "N": 22
}


for coluna, largura in larguras_rank.items():

    ws_rank.column_dimensions[
        coluna
    ].width = largura


ws_rank.freeze_panes = "A8"


# Análise comparativa

titulo(
    ws_comp,
    "Análise Comparativa dos Resultados",
    "Comparação da evolução dos principais indicadores financeiros ao longo dos anos."
)


ws_comp.merge_cells("A3:M4")

ws_comp["A3"] = (
    "A tabela apresenta o comportamento dos principais indicadores financeiros "
    "ao longo dos anos. Para representar o comportamento central dos 1.200 cenários, "
    "foi utilizada a mediana de cada indicador em cada período. A comparação permite "
    "observar mudanças de rentabilidade, liquidez, estrutura de capital e geração "
    "de valor, complementando os resultados encontrados no ranking."
)

ws_comp["A3"].font = Font(
    name="Aptos",
    size=10,
    color=TEXTO
)

ws_comp["A3"].fill = fill_lavanda_clara

ws_comp["A3"].alignment = Alignment(
    vertical="center",
    wrap_text=True
)

ws_comp.row_dimensions[3].height = 25
ws_comp.row_dimensions[4].height = 25


# Tabela comparativa

linha_dados = 6

colunas_comparativo = [
    "Ano",
    "Margem EBITDA",
    "Margem Líquida",
    "LC",
    "LS",
    "LI",
    "LG",
    "PCT",
    "CE",
    "IPL",
    "NOPAT",
    "Capital Investido",
    "EVA"
]


for col, nome in enumerate(
    colunas_comparativo,
    1
):

    ws_comp.cell(
        linha_dados,
        col,
        nome
    )


cabecalho(
    ws_comp,
    linha_dados,
    1,
    len(colunas_comparativo)
)


for linha_excel, (_, registro) in enumerate(
    comparativo_anual.iterrows(),
    linha_dados + 1
):

    for col_excel, nome_coluna in enumerate(
        colunas_comparativo,
        1
    ):

        valor = registro[
            nome_coluna
        ]

        if pd.isna(valor):
            valor = None

        celula = ws_comp.cell(
            linha_excel,
            col_excel,
            valor
        )

        celula.font = Font(
            name="Aptos",
            size=10,
            color=TEXTO
        )

        celula.alignment = Alignment(
            horizontal="center",
            vertical="center"
        )

        celula.border = Border(
            bottom=borda_fina
        )

        if linha_excel % 2 == 0:
            celula.fill = fill_lavanda_clara


        if nome_coluna in [
            "Margem EBITDA",
            "Margem Líquida",
            "CE"
        ]:
            celula.number_format = "0.00%"

        elif nome_coluna in [
            "NOPAT",
            "Capital Investido",
            "EVA"
        ]:
            celula.number_format = '#,##0.00'

        elif nome_coluna != "Ano":
            celula.number_format = "0.00"


larguras_comp = {
    "A": 12,
    "B": 18,
    "C": 18,
    "D": 12,
    "E": 12,
    "F": 12,
    "G": 12,
    "H": 12,
    "I": 12,
    "J": 12,
    "K": 20,
    "L": 22,
    "M": 20
}


for coluna, largura in larguras_comp.items():

    ws_comp.column_dimensions[
        coluna
    ].width = largura


ws_comp.freeze_panes = "B7"


# Dados de base

titulo(
    ws_base,
    "Dados de base",
    "Base tratada utilizada para os cálculos e análises financeiras."
)

cabecalho(
    ws_base,
    4,
    1,
    4
)


ws_base.freeze_panes = "A5"

ws_base.column_dimensions["A"].width = 22
ws_base.column_dimensions["B"].width = 12
ws_base.column_dimensions["C"].width = 48
ws_base.column_dimensions["D"].width = 20


for cell in ws_base["D"][4:]:
    cell.number_format = '#,##0.00'


# Configuração das abas

for ws in [
    ws_dic,
    ws_demo,
    ws_ind,
    ws_rank,
    ws_comp,
    ws_base
]:

    ws.sheet_view.showGridLines = False

    ws.sheet_properties.pageSetUpPr.fitToPage = True

    ws.page_setup.orientation = "landscape"

    ws.page_setup.fitToWidth = 1

    ws.page_setup.fitToHeight = 0


# Abas auxiliares

ws_aux.sheet_state = "hidden"
ws_listas.sheet_state = "hidden"


# Ordem das abas

wb._sheets = [
    ws_dic,
    ws_demo,
    ws_ind,
    ws_rank,
    ws_comp,
    ws_base,
    ws_aux,
    ws_listas
]


# Recálculo

wb.calculation.fullCalcOnLoad = True
wb.calculation.forceFullCalc = True
wb.calculation.calcMode = "auto"


# Salvar

print("Salvando planilha...")

wb.save(
    arquivo_saida
)

print("Planilha criada com sucesso!")
print(arquivo_saida.resolve())

Gravando dados...
Aplicando layout...
Salvando planilha...
Planilha criada com sucesso!
C:\Users\julia\OneDrive - Etec Centro Paula Souza\Documents\GitHub\Projeto1\documentos\entrega 2\contabilidade e finanças\Analise_Financeira.xlsx
